# Archived analysis workflow

This notebook records the author workflow used during revision. It requires the corresponding external model-ready inputs, saved predictions and/or checkpoints. Its `/content/drive/MyDrive` paths describe the author environment and must be reviewed before execution. Do not run all cells merely to inspect the repository. See [analysis/README.md](../README.md) and the main repository README for the supported benchmark entry point.


# SpaGAT R3.5 / R3.7：功能程序后处理与结果核对

**使用普通 CPU 运行时即可，不需要 GPU。挂载 Drive 后按顺序运行全部单元格。**

读取上一轮已完成的目录：
`/content/drive/MyDrive/SpaGAT_revision/R3_5_7_gene_signed_20260928_090940_UTC/`

不训练、不加载 checkpoint、不重新推理，不覆盖现有结果。无需安装额外包；只使用 Colab 已有的 NumPy、pandas 和 Python 标准库。

## 这次完成什么

1. **Program recovery**：在同一批 tumor_1 / tumor_2 validation cells 中比较 observed 和 predicted program scores，报告 PCC、Spearman、MSE 与方向一致性。
2. **Signed program masking**：使用现有 10 次 masking 的逐基因结果和已保存 mean-delta 矩阵，汇总程序变化方向，并与等数量随机屏蔽对照。
3. **结果口径核对**：比较全部 validation receivers 与两类 sender 都存在的 receivers、两种 PCC 数值处理和中位数约定，查找旧 Figure 4 和预处理代码线索。没有充分证据时明确保持未核实。
4. **注释后续分析材料**：导出两类 tumor 的已提供 baseline gene profiles；不自动给 tumor_1/2 命名分子亚型。

## 预先固定的分析规则

- 使用全部 50 个 Human MSigDB Hallmark gene sets，固定版本 **2026.1.Hs**。基因集已嵌入此 notebook，不需要登录或在线下载。[官方说明](https://www.gsea-msigdb.org/gsea/msigdb/collections.jsp)；[官方版本文件](https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2026.1.Hs/h.all.v2026.1.Hs.symbols.gmt)。
- 不依据结果挑选程序。逐项报告本面板覆盖多少基因。至少覆盖 5 个基因才计算；同时标记至少 10 个基因的敏感性子集。覆盖稀少时不能解释为完整通路验证。
- 主分析：在每个 tumor 群体内，以 **observed 的均值和标准差**标准化基因，之后取程序基因平均。预测值使用完全相同的参数，不单独标准化预测值。另报告未标准化均值的敏感性结果。
- 标准化参数来自这批 validation observations，仅用于描述性分析，不构成独立 test 评估。程序方向是相对该群体 observed 平均状态的方向，不是 tumor_1 对 tumor_2 或癌对正常的差异表达。
- signed delta 始终为 masked prediction − full prediction。10 次 SD 仅代表固定模型下的邻居抽样变异。
- 已保存的 cellwise delta 是跨 10 次重复后的均值；代码会将其绝对值与“各重复绝对变化的平均”区分开，不把二者混用。
- 不将 latent channels 当作生物学程序，不做细胞级显著性检验，不声称独立生物学验证或因果作用。

已通过模拟数据检查和本次实际基因/重复表的一致性检查。大型真实预测矩阵只在你的 Drive 上，尚未在这里运行。


## 1. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. 定义后处理程序

直接运行即可。下方固定了上一轮输出目录；如你曾手动移动该目录，只修改 `INPUT` 一行。
原始 9.6 GB CSV 不会重新读入。程序只读取之前导出的数组、小表格、baseline 文件，并在已知项目目录里限量查找代码/历史指标。


In [ ]:
GMT_TEXT = 'HALLMARK_ADIPOGENESIS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ADIPOGENESIS\tABCA1\tABCB8\tACAA2\tACADL\tACADM\tACADS\tACLY\tACO2\tACOX1\tADCY6\tADIG\tADIPOQ\tADIPOR2\tAGPAT3\tAIFM1\tAK2\tALDH2\tALDOA\tANGPT1\tANGPTL4\tAPLP2\tAPOE\tARAF\tARL4A\tATL2\tATP1B3\tATP5PO\tBAZ2A\tBCKDHA\tBCL2L13\tBCL6\tC3\tCAT\tCAVIN1\tCAVIN2\tCCNG2\tCD151\tCD302\tCD36\tCDKN2C\tCHCHD10\tCHUK\tCIDEA\tCMBL\tCMPK1\tCOL15A1\tCOL4A1\tCOQ3\tCOQ5\tCOQ9\tCOX6A1\tCOX7B\tCOX8A\tCPT2\tCRAT\tCS\tCYC1\tCYP4B1\tDBT\tDDT\tDECR1\tDGAT1\tDHCR7\tDHRS7\tDHRS7B\tDLAT\tDLD\tDNAJB9\tDNAJC15\tDRAM2\tECH1\tECHS1\tELMOD3\tELOVL6\tENPP2\tEPHX2\tESRRA\tESYT1\tETFB\tFABP4\tFAH\tFZD4\tG3BP2\tGADD45A\tGBE1\tGHITM\tGPAM\tGPAT4\tGPD2\tGPHN\tGPX3\tGPX4\tGRPEL1\tHADH\tHIBCH\tHSPB8\tIDH1\tIDH3A\tIDH3G\tIFNGR1\tIMMT\tITGA7\tITIH5\tITSN1\tJAGN1\tLAMA4\tLEP\tLIFR\tLIPE\tLPCAT3\tLPL\tLTC4S\tMAP4K3\tMCCC1\tMDH2\tME1\tMGLL\tMGST3\tMIGA2\tMRAP\tMRPL15\tMTARC2\tMTCH2\tMYLK\tNABP1\tNDUFA5\tNDUFAB1\tNDUFB7\tNDUFS3\tNKIRAS1\tNMT1\tOMD\tORM1\tPDCD4\tPEMT\tPEX14\tPFKFB3\tPFKL\tPGM1\tPHLDB1\tPHYH\tPIM3\tPLIN2\tPOR\tPPARG\tPPM1B\tPPP1R15B\tPRDX3\tPREB\tPTCD3\tPTGER3\tQDPR\tRAB34\tREEP5\tREEP6\tRETN\tRETSAT\tRIOK3\tRMDN3\tRNF11\tRREB1\tRTN3\tSAMM50\tSCARB1\tSCP2\tSDHB\tSDHC\tSLC19A1\tSLC1A5\tSLC25A1\tSLC25A10\tSLC27A1\tSLC5A6\tSLC66A3\tSNCG\tSOD1\tSORBS1\tSOWAHC\tSPARCL1\tSQOR\tSSPN\tSTAT5A\tSTOM\tSUCLG1\tSULT1A1\tTALDO1\tTANK\tTKT\tTOB1\tTST\tUBC\tUBQLN1\tUCK1\tUCP2\tUQCR10\tUQCR11\tUQCRC1\tUQCRQ\tVEGFB\tYWHAG\nHALLMARK_ALLOGRAFT_REJECTION\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ALLOGRAFT_REJECTION\tAARS1\tABCE1\tABI1\tACHE\tACVR2A\tAKT1\tAPBB1\tB2M\tBCAT1\tBCL10\tBCL3\tBRCA1\tC2\tCAPG\tCARTPT\tCCL11\tCCL13\tCCL19\tCCL2\tCCL22\tCCL4\tCCL5\tCCL7\tCCND2\tCCND3\tCCR1\tCCR2\tCCR5\tCD1D\tCD2\tCD247\tCD28\tCD3D\tCD3E\tCD3G\tCD4\tCD40\tCD40LG\tCD47\tCD7\tCD74\tCD79A\tCD80\tCD86\tCD8A\tCD8B\tCD96\tCDKN2A\tCFP\tCRTAM\tCSF1\tCSK\tCTSS\tCXCL13\tCXCL9\tCXCR3\tDARS1\tDEGS1\tDYRK3\tEGFR\tEIF3A\tEIF3D\tEIF3J\tEIF4G3\tEIF5A\tELANE\tELF4\tEREG\tETS1\tF2\tF2R\tFAS\tFASLG\tFCGR2B\tFGR\tFLNA\tFYB1\tGALNT1\tGBP2\tGCNT1\tGLMN\tGPR65\tGZMA\tGZMB\tHCLS1\tHDAC9\tHIF1A\tHLA-A\tHLA-DMA\tHLA-DMB\tHLA-DOA\tHLA-DOB\tHLA-DQA1\tHLA-DRA\tHLA-E\tHLA-G\tICAM1\tICOSLG\tIFNAR2\tIFNG\tIFNGR1\tIFNGR2\tIGSF6\tIKBKB\tIL10\tIL11\tIL12A\tIL12B\tIL12RB1\tIL13\tIL15\tIL16\tIL18\tIL18RAP\tIL1B\tIL2\tIL27RA\tIL2RA\tIL2RB\tIL2RG\tIL4\tIL4R\tIL6\tIL7\tIL9\tINHBA\tINHBB\tIRF4\tIRF7\tIRF8\tITGAL\tITGB2\tITK\tJAK2\tKLRD1\tKRT1\tLCK\tLCP2\tLIF\tLTB\tLY75\tLY86\tLYN\tMAP3K7\tMAP4K1\tMBL2\tMMP9\tMRPL3\tMTIF2\tNCF4\tNCK1\tNCR1\tNLRP3\tNME1\tNOS2\tNPM1\tPF4\tPRF1\tPRKCB\tPRKCG\tPSMB10\tPTPN6\tPTPRC\tRARS1\tRIPK2\tRPL39\tRPL3L\tRPL9\tRPS19\tRPS3A\tRPS9\tSIT1\tSOCS1\tSOCS5\tSPI1\tSRGN\tST8SIA4\tSTAB1\tSTAT1\tSTAT4\tTAP1\tTAP2\tTAPBP\tTGFB1\tTGFB2\tTHY1\tTIMP1\tTLR1\tTLR2\tTLR3\tTLR6\tTNF\tTPD52\tTRAF2\tTRAT1\tUBE2D1\tUBE2N\tWARS1\tWAS\tZAP70\nHALLMARK_ANDROGEN_RESPONSE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ANDROGEN_RESPONSE\tABCC4\tABHD2\tACSL3\tACTN1\tADAMTS1\tADRM1\tAKAP12\tAKT1\tALDH1A3\tANKH\tAPPBP2\tARID5B\tAZGP1\tB2M\tB4GALT1\tBMPR1B\tCAMKK2\tCCND1\tCCND3\tCDC14B\tCDK6\tCENPN\tDBI\tDHCR24\tDNAJB9\tELK4\tELL2\tELOVL5\tFADS1\tFKBP5\tGNAI3\tGPD1L\tGSR\tGUCY1A1\tH1-0\tHERC3\tHMGCR\tHMGCS1\tHOMER2\tHPGD\tHSD17B14\tIDI1\tINPP4B\tINSIG1\tIQGAP2\tITGAV\tKLK2\tKLK3\tKRT19\tKRT8\tLIFR\tLMAN1\tMAF\tMAK\tMAP7\tMERTK\tMYL12A\tNCOA4\tNDRG1\tNGLY1\tNKX3-1\tPA2G4\tPDLIM5\tPGM3\tPIAS1\tPLPP1\tPMEPA1\tPTK2B\tPTPN21\tRAB4A\tRPS6KA3\tRRP12\tSAT1\tSCD\tSEC24D\tSELENOP\tSGK1\tSLC26A2\tSLC38A2\tSMS\tSORD\tSPCS3\tSPDEF\tSRF\tSRP19\tSTEAP4\tSTK39\tTARP\tTMEM50A\tTMPRSS2\tTNFAIP8\tTPD52\tTSC22D1\tUAP1\tUBE2I\tUBE2J1\tVAPA\tXRCC5\tXRCC6\tZBTB10\tZMIZ1\nHALLMARK_ANGIOGENESIS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ANGIOGENESIS\tAPOH\tAPP\tCCND2\tCOL3A1\tCOL5A2\tCXCL6\tFGFR1\tFSTL1\tITGAV\tJAG1\tJAG2\tKCNJ8\tLPL\tLRPAP1\tLUM\tMSX1\tNRP1\tOLR1\tPDGFA\tPF4\tPGLYRP1\tPOSTN\tPRG2\tPTK2\tS100A4\tSERPINA5\tSLCO2A1\tSPP1\tSTC1\tTHBD\tTIMP1\tTNFRSF21\tVAV2\tVCAN\tVEGFA\tVTN\nHALLMARK_APICAL_JUNCTION\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_APICAL_JUNCTION\tACTA1\tACTB\tACTC1\tACTG1\tACTG2\tACTN1\tACTN2\tACTN3\tACTN4\tADAM15\tADAM23\tADAM9\tADAMTS5\tADRA1B\tAKT2\tAKT3\tALOX15B\tAMH\tAMIGO1\tAMIGO2\tARHGEF6\tARPC2\tATP1A3\tB4GALT1\tBAIAP2\tBMP1\tCADM2\tCADM3\tCALB2\tCAP1\tCD209\tCD274\tCD276\tCD34\tCD86\tCD99\tCDH1\tCDH11\tCDH15\tCDH3\tCDH4\tCDH6\tCDH8\tCDK8\tCDSN\tCERCAM\tCLDN11\tCLDN14\tCLDN15\tCLDN18\tCLDN19\tCLDN4\tCLDN5\tCLDN6\tCLDN7\tCLDN8\tCLDN9\tCNN2\tCNTN1\tCOL16A1\tCOL17A1\tCOL9A1\tCRAT\tCRB3\tCTNNA1\tCTNND1\tCX3CL1\tDHX16\tDLG1\tDMP1\tDSC1\tDSC3\tEGFR\tEPB41L2\tEVL\tEXOC4\tFBN1\tFLNC\tFSCN1\tFYB1\tGAMT\tGNAI1\tGNAI2\tGRB7\tGTF2F1\tHADH\tHRAS\tICAM1\tICAM2\tICAM4\tICAM5\tIKBKG\tINPPL1\tINSIG1\tIRS1\tITGA10\tITGA2\tITGA3\tITGA9\tITGB1\tITGB4\tJAM3\tJUP\tKCNH2\tKRT31\tLAMA3\tLAMB3\tLAMC2\tLAYN\tLDLRAP1\tLIMA1\tMADCAM1\tMAP3K20\tMAP4K2\tMAPK11\tMAPK13\tMAPK14\tMDK\tMMP2\tMMP9\tMPZL1\tMPZL2\tMSN\tMVD\tMYH10\tMYH9\tMYL12B\tMYL9\tNECTIN1\tNECTIN2\tNECTIN3\tNECTIN4\tNEGR1\tNEXN\tNF1\tNF2\tNFASC\tNHERF4\tNLGN2\tNLGN3\tNRAP\tNRTN\tNRXN2\tPALS1\tPARD6G\tPARVA\tPBX2\tPCDH1\tPECAM1\tPFN1\tPIK3CB\tPIK3R3\tPKD1\tPLCG1\tPPP2R2C\tPTEN\tPTK2\tPTPRC\tRAC2\tRASA1\tRHOF\tRRAS\tRSU1\tSDC3\tSGCE\tSHC1\tSHROOM2\tSIRPA\tSKAP2\tSLC30A3\tSLIT2\tSORBS3\tSPEG\tSRC\tSTX4\tSYK\tSYMPK\tTAOK2\tTGFBI\tTHBS3\tTHY1\tTIAL1\tTJP1\tTMEM8B\tTNFRSF11B\tTRAF1\tTRO\tTSC1\tTSPAN4\tTUBG1\tVASP\tVAV2\tVCAM1\tVCAN\tVCL\tVWF\tWASL\tWNK4\tYWHAH\tZYX\nHALLMARK_APICAL_SURFACE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_APICAL_SURFACE\tADAM10\tADIPOR2\tAFAP1L2\tAKAP7\tAPP\tATP6V0A4\tATP8B1\tB4GALT1\tBRCA1\tCD160\tCROCC\tCRYBG1\tCX3CL1\tDCBLD2\tEFNA5\tEPHB4\tFLOT2\tGAS1\tGATA3\tGHRL\tGSTM3\tHSPB1\tIL2RB\tIL2RG\tLYN\tLYPD3\tMAL\tMDGA1\tNCOA6\tNTNG1\tPCSK9\tPKHD1\tPLAUR\tRHCG\tRTN4RL1\tSCUBE1\tSHROOM2\tSLC22A12\tSLC2A4\tSLC34A3\tSRPX\tSULF2\tTHY1\tTMEM8B\nHALLMARK_APOPTOSIS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_APOPTOSIS\tADD1\tAIFM3\tANKH\tANXA1\tAPP\tATF3\tAVPR1A\tBAX\tBCAP31\tBCL10\tBCL2L1\tBCL2L10\tBCL2L11\tBCL2L2\tBGN\tBID\tBIK\tBIRC3\tBMF\tBMP2\tBNIP3L\tBRCA1\tBTG2\tBTG3\tCASP1\tCASP2\tCASP3\tCASP4\tCASP6\tCASP7\tCASP8\tCASP9\tCAV1\tCCNA1\tCCND1\tCCND2\tCD14\tCD2\tCD38\tCD44\tCD69\tCDC25B\tCDK2\tCDKN1A\tCDKN1B\tCFLAR\tCLU\tCREBBP\tCTH\tCTNNB1\tCYLD\tDAP\tDAP3\tDCN\tDDIT3\tDFFA\tDIABLO\tDNAJA1\tDNAJC3\tDNM1L\tDPYD\tEBP\tEGR3\tEMP1\tENO2\tERBB2\tERBB3\tEREG\tETF1\tF2\tF2R\tFAS\tFASLG\tFDXR\tFEZ1\tGADD45A\tGADD45B\tGCH1\tGNA15\tGPX1\tGPX3\tGPX4\tGSN\tGSR\tGSTM1\tGUCY2D\tH1-0\tHGF\tHMGB2\tHMOX1\tHSPB1\tIER3\tIFITM3\tIFNB1\tIFNGR1\tIGF2R\tIGFBP6\tIL18\tIL1A\tIL1B\tIL6\tIRF1\tISG20\tJUN\tKRT18\tLEF1\tLGALS3\tLMNA\tLUM\tMADD\tMCL1\tMGMT\tMMP2\tNEDD9\tNEFH\tPAK1\tPDCD4\tPDGFRB\tPEA15\tPLAT\tPLCB2\tPLPPR4\tPMAIP1\tPPP2R5B\tPPP3R1\tPPT1\tPRF1\tPSEN1\tPSEN2\tPTK2\tRARA\tRELA\tRETSAT\tRHOB\tRHOT2\tRNASEL\tROCK1\tSAT1\tSATB1\tSC5D\tSLC20A1\tSMAD7\tSOD1\tSOD2\tSPTAN1\tSQSTM1\tTAP1\tTGFB2\tTGFBR3\tTIMP1\tTIMP2\tTIMP3\tTNF\tTNFRSF12A\tTNFSF10\tTOP2A\tTSPO\tTXNIP\tVDAC2\tWEE1\tXIAP\nHALLMARK_BILE_ACID_METABOLISM\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_BILE_ACID_METABOLISM\tABCA1\tABCA2\tABCA3\tABCA4\tABCA5\tABCA6\tABCA8\tABCA9\tABCD1\tABCD2\tABCD3\tABCG4\tABCG8\tACSL1\tACSL5\tAGXT\tAKR1D1\tALDH1A1\tALDH8A1\tALDH9A1\tAMACR\tAPOA1\tAQP9\tAR\tATXN1\tBBOX1\tBCAR3\tBMP6\tCAT\tCH25H\tCROT\tCYP27A1\tCYP39A1\tCYP46A1\tCYP7A1\tCYP7B1\tCYP8B1\tDHCR24\tDIO1\tDIO2\tEFHC1\tEPHX2\tFADS1\tFADS2\tFDXR\tGC\tGCLM\tGNMT\tGNPAT\tGSTK1\tHACL1\tHAO1\tHSD17B11\tHSD17B4\tHSD17B6\tHSD3B1\tHSD3B7\tIDH1\tIDH2\tIDI1\tISOC1\tKLF1\tLCK\tLIPE\tLONP2\tMLYCD\tNEDD4\tNPC1\tNR0B2\tNR1H4\tNR1I2\tNR3C2\tNUDT12\tOPTN\tPAOX\tPECR\tPEX1\tPEX11A\tPEX11G\tPEX12\tPEX13\tPEX16\tPEX19\tPEX26\tPEX6\tPEX7\tPFKM\tPHYH\tPIPOX\tPNPLA8\tPRDX5\tPXMP2\tRBP1\tRETSAT\tRXRA\tRXRG\tSCP2\tSERPINA6\tSLC23A1\tSLC23A2\tSLC27A2\tSLC27A5\tSLC29A1\tSLC35B2\tSLC67A1\tSLCO1A2\tSOAT2\tSOD1\tSULT1B1\tSULT2B1\tTFCP2L1\tTTR\nHALLMARK_CHOLESTEROL_HOMEOSTASIS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_CHOLESTEROL_HOMEOSTASIS\tABCA2\tACAT2\tACSS2\tACTG1\tADH4\tALCAM\tALDOC\tANTXR2\tANXA13\tANXA5\tATF3\tATF5\tATXN2\tAVPR1A\tCBS\tCD9\tCHKA\tCLU\tCPEB2\tCTNNB1\tCXCL16\tCYP51A1\tDHCR7\tEBP\tECH1\tERRFI1\tETHE1\tFABP5\tFADS2\tFASN\tFBXO6\tFDFT1\tFDPS\tGLDC\tGNAI1\tGPX8\tGSTM2\tGUSB\tHMGCR\tHMGCS1\tHSD17B7\tIDI1\tJAG1\tLDLR\tLGALS3\tLGMN\tLPL\tLSS\tMAL2\tMVD\tMVK\tNFIL3\tNIBAN1\tNSDHL\tPCYT2\tPDK3\tPLAUR\tPLSCR1\tPMVK\tPNRC1\tPPARG\tS100A11\tSC5D\tSCD\tSEMA3B\tSQLE\tSREBF2\tSTARD4\tSTX5\tTM7SF2\tTMEM97\tTNFRSF12A\tTP53INP1\tTRIB3\nHALLMARK_COAGULATION\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_COAGULATION\tA2M\tACOX2\tADAM9\tANG\tANXA1\tAPOA1\tAPOC1\tAPOC2\tAPOC3\tARF4\tBMP1\tC1QA\tC1R\tC1S\tC2\tC3\tC8A\tC8B\tC8G\tC9\tCAPN2\tCAPN5\tCASP9\tCD9\tCFB\tCFD\tCFH\tCFI\tCLU\tCOMP\tCPB2\tCPN1\tCPQ\tCRIP2\tCSRP1\tCTSB\tCTSE\tCTSH\tCTSK\tCTSO\tCTSV\tDCT\tDPP4\tDUSP14\tDUSP6\tF10\tF11\tF12\tF13B\tF2\tF2RL2\tF3\tF8\tF9\tFBN1\tFGA\tFGG\tFN1\tFURIN\tFYN\tGDA\tGNB2\tGNG12\tGP1BA\tGP9\tGSN\tHMGCS2\tHNF4A\tHPN\tHRG\tHTRA1\tISCU\tITGA2\tITGB3\tITIH1\tKLF7\tKLK8\tKLKB1\tLAMP2\tLEFTY2\tLGMN\tLRP1\tLTA4H\tMAFF\tMASP2\tMBL2\tMEP1A\tMMP1\tMMP10\tMMP11\tMMP14\tMMP15\tMMP2\tMMP3\tMMP7\tMMP8\tMMP9\tMSRB2\tMST1\tOLR1\tP2RY1\tPDGFB\tPECAM1\tPEF1\tPF4\tPLAT\tPLAU\tPLEK\tPLG\tPREP\tPROC\tPROS1\tPROZ\tPRSS23\tRABIF\tRAC1\tRAPGEF3\tRGN\tS100A1\tS100A13\tSERPINA1\tSERPINB2\tSERPINC1\tSERPINE1\tSERPING1\tSH2B2\tSIRT2\tSPARC\tTF\tTFPI2\tTHBD\tTHBS1\tTIMP1\tTIMP3\tTMPRSS6\tUSP11\tVWF\tWDR1\nHALLMARK_COMPLEMENT\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_COMPLEMENT\tACTN2\tADAM9\tADRA2B\tAKAP10\tANG\tANXA5\tAPOA4\tAPOBEC3F\tAPOBEC3G\tAPOC1\tATOX1\tBRPF3\tC1QA\tC1QC\tC1R\tC1S\tC2\tC3\tC4BPB\tC9\tCA2\tCALM1\tCALM3\tCASP1\tCASP10\tCASP3\tCASP4\tCASP5\tCASP7\tCASP9\tCBLB\tCCL5\tCD36\tCD40LG\tCD46\tCD55\tCD59\tCDA\tCDH13\tCDK5R1\tCEBPB\tCFB\tCFH\tCLU\tCOL4A2\tCP\tCPM\tCPQ\tCR1\tCR2\tCSRP1\tCTSB\tCTSC\tCTSD\tCTSH\tCTSL\tCTSO\tCTSS\tCTSV\tCXCL1\tDGKG\tDGKH\tDOCK10\tDOCK4\tDOCK9\tDPP4\tDUSP5\tDUSP6\tDYRK2\tEHD1\tERAP2\tF10\tF2\tF3\tF5\tF7\tF8\tFCER1G\tFCN1\tFDX1\tFN1\tFYN\tGATA3\tGCA\tGMFB\tGNAI2\tGNAI3\tGNB2\tGNB4\tGNG2\tGNGT2\tGP1BA\tGP9\tGPD2\tGRB2\tGZMA\tGZMB\tGZMK\tHNF4A\tHPCAL4\tHSPA1A\tHSPA5\tIL6\tIRF1\tIRF2\tIRF7\tITGAM\tITIH1\tJAK2\tKCNIP2\tKCNIP3\tKIF2A\tKLK1\tKLKB1\tKYNU\tL3MBTL4\tLAMP2\tLAP3\tLCK\tLCP2\tLGALS3\tLGMN\tLIPA\tLRP1\tLTA4H\tLTF\tLYN\tMAFF\tME1\tMMP12\tMMP13\tMMP14\tMMP15\tMMP8\tMSRB1\tMT3\tNOTCH4\tOLR1\tPCLO\tPCSK9\tPDGFB\tPDP1\tPFN1\tPHEX\tPIK3CA\tPIK3CG\tPIK3R5\tPIM1\tPLA2G4A\tPLA2G7\tPLAT\tPLAUR\tPLEK\tPLG\tPLSCR1\tPPP2CB\tPPP4C\tPRCP\tPRDM4\tPREP\tPRKCD\tPRSS3\tPRSS36\tPSEN1\tPSMB9\tRABIF\tRAF1\tRASGRP1\tRBSN\tRCE1\tRHOG\tRNF4\tS100A12\tS100A13\tS100A9\tSCG3\tSERPINA1\tSERPINB2\tSERPINC1\tSERPINE1\tSERPING1\tSH2B3\tSIRT6\tSPOCK2\tSRC\tSTX4\tTFPI2\tTIMP1\tTIMP2\tTMPRSS6\tTNFAIP3\tUSP14\tUSP15\tUSP16\tUSP8\tVCPIP1\tWAS\tXPNPEP1\tZEB1\tZFPM2\nHALLMARK_DNA_REPAIR\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_DNA_REPAIR\tAAAS\tADA\tADCY6\tADRM1\tAGO4\tAK1\tAK3\tALYREF\tAPRT\tARL6IP1\tBCAM\tBCAP31\tBOLA2\tBRF2\tCANT1\tCCNO\tCDA\tCETN2\tCLP1\tCMPK2\tCOX17\tCSTF3\tDAD1\tDCTN4\tDDB1\tDDB2\tDGCR8\tDGUOK\tDUT\tEDF1\tEIF1B\tELL\tELOA\tERCC1\tERCC2\tERCC3\tERCC4\tERCC5\tERCC8\tFEN1\tGMPR2\tGPX4\tGSDME\tGTF2A2\tGTF2B\tGTF2F1\tGTF2H1\tGTF2H3\tGTF2H5\tGTF3C5\tGUK1\tHCLS1\tHPRT1\tIMPDH2\tITPA\tLIG1\tMPC2\tMPG\tMRPL40\tNCBP2\tNELFB\tNELFCD\tNELFE\tNFX1\tNME1\tNME3\tNME4\tNPR2\tNT5C\tNT5C3A\tNUDT21\tNUDT9\tPCNA\tPDE4B\tPDE6G\tPNP\tPOLA1\tPOLA2\tPOLB\tPOLD1\tPOLD3\tPOLD4\tPOLE4\tPOLH\tPOLL\tPOLR1C\tPOLR1D\tPOLR1H\tPOLR2A\tPOLR2C\tPOLR2D\tPOLR2E\tPOLR2F\tPOLR2G\tPOLR2H\tPOLR2I\tPOLR2J\tPOLR2K\tPOLR3C\tPOLR3GL\tPOM121\tPRIM1\tRAD51\tRAD52\tRAE1\tRALA\tRBX1\tREV3L\tRFC2\tRFC3\tRFC4\tRFC5\tRNMT\tRPA2\tRPA3\tRRM2B\tSAC3D1\tSDCBP\tSEC61A1\tSF3A3\tSMAD5\tSNAPC4\tSNAPC5\tSRSF6\tSSRP1\tSTX3\tSUPT4H1\tSUPT5H\tSURF1\tTAF10\tTAF12\tTAF13\tTAF1C\tTAF6\tTAF9\tTARBP2\tTK2\tTMED2\tTP53\tTSG101\tTYMS\tUMPS\tUPF3B\tUSP11\tVPS28\tVPS37B\tVPS37D\tXPC\tZNF707\tZWINT\nHALLMARK_E2F_TARGETS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_E2F_TARGETS\tAK2\tANP32E\tASF1A\tASF1B\tATAD2\tAURKA\tAURKB\tBARD1\tBIRC5\tBRCA1\tBRCA2\tBRMS1L\tBUB1B\tCBX5\tCCNB2\tCCNE1\tCCP110\tCDC20\tCDC25A\tCDC25B\tCDCA3\tCDCA8\tCDK1\tCDK4\tCDKN1A\tCDKN1B\tCDKN2A\tCDKN2C\tCDKN3\tCENPE\tCENPM\tCHEK1\tCHEK2\tCIT\tCKS1B\tCKS2\tCNOT9\tCSE1L\tCTCF\tCTPS1\tDCK\tDCLRE1B\tDCTPP1\tDDX39A\tDEK\tDEPDC1\tDIAPH3\tDLGAP5\tDNMT1\tDONSON\tDSCC1\tDUT\tE2F8\tEED\tEIF2S1\tESPL1\tEXOSC8\tEZH2\tGINS1\tGINS3\tGINS4\tGSPT1\tH2AX\tH2AZ1\tHELLS\tHMGA1\tHMGB2\tHMGB3\tHMMR\tHNRNPD\tHUS1\tILF3\tING3\tIPO7\tJPT1\tKIF18B\tKIF22\tKIF2C\tKIF4A\tKPNA2\tLBR\tLIG1\tLMNB1\tLUC7L3\tLYAR\tMAD2L1\tMCM2\tMCM3\tMCM4\tMCM5\tMCM6\tMCM7\tMELK\tMKI67\tMLH1\tMMS22L\tMRE11\tMSH2\tMTHFD2\tMXD3\tMYBL2\tMYC\tNAA38\tNAP1L1\tNASP\tNBN\tNCAPD2\tNME1\tNOLC1\tNOP56\tNUDT21\tNUP107\tNUP153\tNUP205\tORC2\tORC6\tPA2G4\tPAICS\tPAN2\tPCNA\tPDS5B\tPHF5A\tPLK1\tPLK4\tPMS2\tPNN\tPOLA2\tPOLD1\tPOLD2\tPOLD3\tPOLE\tPOLE4\tPOP7\tPPM1D\tPPP1R8\tPRDX4\tPRIM2\tPRKDC\tPRPS1\tPSIP1\tPSMC3IP\tPTTG1\tRACGAP1\tRAD1\tRAD21\tRAD50\tRAD51AP1\tRAD51C\tRAN\tRANBP1\tRBBP7\tRFC1\tRFC2\tRFC3\tRNASEH2A\tRPA1\tRPA2\tRPA3\tRRM2\tSHMT1\tSLBP\tSMC1A\tSMC3\tSMC4\tSMC6\tSNRPB\tSPAG5\tSPC24\tSPC25\tSRSF1\tSRSF2\tSSRP1\tSTAG1\tSTMN1\tSUV39H1\tSYNCRIP\tTACC3\tTBRG4\tTCF19\tTFRC\tTIMELESS\tTIPIN\tTK1\tTMPO\tTOP2A\tTP53\tTRA2B\tTRIP13\tTUBB\tTUBG1\tUBE2S\tUBE2T\tUBR7\tUNG\tUSP1\tWDR90\tWEE1\tXPO1\tXRCC6\tZW10\nHALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION\tABI3BP\tACTA2\tADAM12\tANPEP\tAPLP1\tAREG\tBASP1\tBDNF\tBGN\tBMP1\tCADM1\tCALD1\tCALU\tCAP2\tCAPG\tCCN1\tCCN2\tCD44\tCD59\tCDH11\tCDH2\tCDH6\tCOL11A1\tCOL12A1\tCOL16A1\tCOL1A1\tCOL1A2\tCOL3A1\tCOL4A1\tCOL4A2\tCOL5A1\tCOL5A2\tCOL5A3\tCOL6A2\tCOL6A3\tCOL7A1\tCOL8A2\tCOLGALT1\tCOMP\tCOPA\tCRLF1\tCTHRC1\tCXCL1\tCXCL12\tCXCL6\tCXCL8\tDAB2\tDCN\tDKK1\tDPYSL3\tDST\tECM1\tECM2\tEDIL3\tEFEMP2\tELN\tEMP3\tENO2\tFAP\tFAS\tFBLN1\tFBLN2\tFBLN5\tFBN1\tFBN2\tFERMT2\tFGF2\tFLNA\tFMOD\tFN1\tFOXC2\tFSTL1\tFSTL3\tFUCA1\tFZD8\tGADD45A\tGADD45B\tGAS1\tGEM\tGJA1\tGLIPR1\tGPC1\tGPX7\tGREM1\tHTRA1\tID2\tIGFBP2\tIGFBP3\tIGFBP4\tIL15\tIL32\tIL6\tINHBA\tITGA2\tITGA5\tITGAV\tITGB1\tITGB3\tITGB5\tJUN\tLAMA1\tLAMA2\tLAMA3\tLAMC1\tLAMC2\tLGALS1\tLOX\tLOXL1\tLOXL2\tLRP1\tLRRC15\tLUM\tMAGEE1\tMATN2\tMATN3\tMCM7\tMEST\tMFAP5\tMGP\tMMP1\tMMP14\tMMP2\tMMP3\tMSX1\tMXRA5\tMYL9\tMYLK\tNID2\tNNMT\tNOTCH2\tNT5E\tNTM\tOXTR\tP3H1\tPCOLCE\tPCOLCE2\tPDGFRB\tPDLIM4\tPFN2\tPLAUR\tPLOD1\tPLOD2\tPLOD3\tPMEPA1\tPMP22\tPOSTN\tPPIB\tPRRX1\tPRSS2\tPTHLH\tPTX3\tPVR\tQSOX1\tRGS4\tRHOB\tSAT1\tSCG2\tSDC1\tSDC4\tSERPINE1\tSERPINE2\tSERPINH1\tSFRP1\tSFRP4\tSGCB\tSGCD\tSGCG\tSLC6A8\tSLIT2\tSLIT3\tSNAI2\tSNTB1\tSPARC\tSPOCK1\tSPP1\tTAGLN\tTFPI2\tTGFB1\tTGFBI\tTGFBR3\tTGM2\tTHBS1\tTHBS2\tTHY1\tTIMP1\tTIMP3\tTNC\tTNFAIP3\tTNFRSF11B\tTNFRSF12A\tTPM1\tTPM2\tTPM4\tVCAM1\tVCAN\tVEGFA\tVEGFC\tVIM\tWIPF1\tWNT5A\nHALLMARK_ESTROGEN_RESPONSE_EARLY\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ESTROGEN_RESPONSE_EARLY\tABAT\tABCA3\tABHD2\tABLIM1\tADCY1\tADCY9\tADD3\tAFF1\tAKAP1\tALDH3B1\tAMFR\tANXA9\tAQP3\tAR\tAREG\tARL3\tASB13\tB4GALT1\tBAG1\tBCL11B\tBCL2\tBHLHE40\tBLVRB\tCA12\tCALB2\tCALCR\tCANT1\tCBFA2T3\tCCN5\tCCND1\tCD44\tCELSR1\tCELSR2\tCHPT1\tCISH\tCLDN7\tCLIC3\tCXCL12\tCYP26B1\tDEPTOR\tDHCR7\tDHRS2\tDHRS3\tDLC1\tDYNLT3\tEEIG1\tEGR3\tELF1\tELF3\tELOVL2\tELOVL5\tENDOD1\tESRP2\tFARP1\tFASN\tFCMR\tFDFT1\tFHL2\tFKBP4\tFKBP5\tFLNB\tFOS\tFOXC1\tFRK\tGAB2\tGFRA1\tGJA1\tGLA\tGREB1\tHES1\tHR\tHSPB8\tIGF1R\tIGFBP4\tIL17RB\tIL6ST\tINHBB\tINPP5F\tISG20L2\tITPK1\tJAK2\tKAZN\tKCNK15\tKCNK5\tKDM4B\tKLF10\tKLF4\tKLK10\tKRT13\tKRT15\tKRT18\tKRT19\tKRT8\tLAD1\tLRIG1\tMAPT\tMAST4\tMED13L\tMED24\tMICB\tMINDY1\tMLPH\tMPPED2\tMREG\tMSMB\tMUC1\tMYB\tMYBBP1A\tMYBL1\tMYC\tMYOF\tNADSYN1\tNAV2\tNBL1\tNCOR2\tNHERF1\tNPY1R\tNRIP1\tNXT1\tOLFM1\tOLFML3\tOPN3\tOVOL2\tP2RY2\tPAPSS2\tPDLIM3\tPDZK1\tPEX11A\tPGR\tPLAAT3\tPMAIP1\tPODXL\tPPIF\tPRSS23\tPTGES\tRAB17\tRAB31\tRAPGEFL1\tRARA\tRASGRP1\tRBBP8\tREEP1\tRET\tRETREG1\tRHOBTB3\tRHOD\tRPS6KA2\tRRP12\tSCARB1\tSCNN1A\tSEC14L2\tSEMA3B\tSFN\tSH3BP5\tSIAH2\tSLC16A1\tSLC19A2\tSLC1A1\tSLC1A4\tSLC22A5\tSLC24A3\tSLC26A2\tSLC27A2\tSLC2A1\tSLC37A1\tSLC39A6\tSLC7A2\tSLC7A5\tSNX24\tSOX3\tSTC2\tSULT2B1\tSVIL\tSYBU\tSYNGR1\tSYT12\tTBC1D30\tTFAP2C\tTFF1\tTFF3\tTGIF2\tTGM2\tTHSD4\tTIAM1\tTIPARP\tTJP3\tTMEM164\tTMPRSS3\tTOB1\tTPBG\tTPD52L1\tTSKU\tTTC39A\tTUBB2B\tUGCG\tUNC119\tWFS1\tWWC1\tXBP1\tZNF185\nHALLMARK_ESTROGEN_RESPONSE_LATE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ESTROGEN_RESPONSE_LATE\tABCA3\tABHD2\tACOX2\tADD3\tAFF1\tAGR2\tALDH3A2\tALDH3B1\tAMFR\tANXA9\tAREG\tARL3\tASCL1\tASS1\tATP2B4\tBAG1\tBATF\tBCL2\tBLVRB\tBTG3\tCA12\tCA2\tCACNA2D2\tCALCR\tCAV1\tCCN5\tCCNA1\tCCND1\tCD44\tCD9\tCDC20\tCDC6\tCDH1\tCELSR2\tCHPT1\tCHST8\tCISH\tCKB\tCLIC3\tCOX6C\tCPE\tCXCL12\tCXCL14\tCYP26B1\tCYP4F11\tDCXR\tDHCR7\tDHRS2\tDLG5\tDNAJC1\tDNAJC12\tDUSP2\tDYNLT3\tEEIG1\tEGR3\tELOVL5\tEMP2\tETFB\tFABP5\tFARP1\tFDFT1\tFGFR3\tFKBP4\tFKBP5\tFLNB\tFOS\tFOXC1\tFRK\tGAL\tGALE\tGFUS\tGINS2\tGJB3\tGLA\tGPER1\tHMGCS2\tHOMER2\tHPRT1\tHR\tHSPA4L\tHSPB8\tID2\tIDH2\tIGFBP4\tIGSF1\tIL17RB\tIL6ST\tIMPA2\tISG20\tITPK1\tJAK1\tJAK2\tKCNK5\tKIF20A\tKLF4\tKLK10\tKLK11\tKRT13\tKRT19\tLAMC2\tLARGE1\tLLGL2\tLSR\tLTF\tMAPK13\tMAPT\tMDK\tMEST\tMETTL3\tMICB\tMOCS2\tMYB\tMYOF\tNAB2\tNBL1\tNCOR2\tNHERF1\tNMU\tNPY1R\tNRIP1\tNXT1\tOLFM1\tOPN3\tOVOL2\tPAPSS2\tPCP4\tPDCD4\tPDLIM3\tPDZK1\tPERP\tPGR\tPKP3\tPLAAT3\tPLAC1\tPLK4\tPLXNB1\tPPIF\tPRKAR2B\tPRLR\tPRSS23\tPTGER3\tPTGES\tPTPN6\tRAB31\tRABEP1\tRAPGEFL1\tRBBP8\tRET\tRNASEH2A\tRPS6KA2\tS100A9\tSCARB1\tSCNN1A\tSCUBE2\tSEMA3B\tSERPINA1\tSERPINA3\tSERPINA5\tSFN\tSGK1\tSIAH2\tSLC16A1\tSLC1A4\tSLC22A5\tSLC24A3\tSLC26A2\tSLC27A2\tSLC29A1\tSLC2A8\tSLC7A5\tSNX10\tSORD\tSOX3\tST14\tST6GALNAC2\tSTIL\tSULT2B1\tTFAP2C\tTFF1\tTFF3\tTFPI2\tTH\tTIAM1\tTJP3\tTMPRSS3\tTNNC1\tTOB1\tTOP2A\tTPBG\tTPD52L1\tTPSAB1\tTRIM29\tTSPAN13\tTST\tUGDH\tUNC13B\tWFS1\tXBP1\tXRCC3\tZFP36\nHALLMARK_FATTY_ACID_METABOLISM\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_FATTY_ACID_METABOLISM\tAADAT\tACAA1\tACAA2\tACADL\tACADM\tACADS\tACADVL\tACAT2\tACO2\tACOT2\tACOT8\tACOX1\tACSL1\tACSL4\tACSL5\tACSM3\tACSS1\tADH1C\tADH7\tADIPOR2\tADSL\tALAD\tALDH1A1\tALDH3A1\tALDH3A2\tALDH9A1\tALDOA\tAOC3\tAPEX1\tAQP7\tAUH\tBCKDHB\tBLVRA\tBMPR1B\tBPHL\tCA2\tCA4\tCA6\tCBR1\tCBR3\tCD1D\tCD36\tCEL\tCIDEA\tCPOX\tCPT1A\tCPT2\tCRAT\tCRYZ\tCYP1A1\tCYP4A11\tCYP4A22\tD2HGDH\tDECR1\tDHCR24\tDLD\tDLST\tECH1\tECHS1\tECI1\tECI2\tEHHADH\tELOVL5\tENO2\tENO3\tEPHX1\tERP29\tETFDH\tFABP1\tFABP2\tFASN\tFH\tFMO1\tG0S2\tGABARAPL1\tGAD2\tGAPDHS\tGCDH\tGLUL\tGPD1\tGPD2\tGRHPR\tGSTZ1\tH2AZ1\tHADH\tHADHB\tHAO2\tHCCS\tHIBCH\tHMGCL\tHMGCS1\tHMGCS2\tHPGD\tHSD17B10\tHSD17B11\tHSD17B4\tHSD17B7\tHSDL2\tHSP90AA1\tHSPH1\tIDH1\tIDH3B\tIDH3G\tIDI1\tIL4I1\tINMT\tKMT5A\tLDHA\tLGALS1\tLTC4S\tMAOA\tMCEE\tMDH1\tMDH2\tME1\tMETAP1\tMGLL\tMIF\tMIX23\tMLYCD\tNBN\tNCAPH2\tNSDHL\tNTHL1\tODC1\tOSTC\tPCBD1\tPDHA1\tPDHB\tPPARA\tPRDX6\tPSME1\tPTPRG\tPTS\tRAP1GDS1\tRDH11\tRDH16\tREEP6\tRETSAT\tS100A10\tSDHA\tSDHC\tSDHD\tSERINC1\tSLC22A5\tSMS\tSUCLA2\tSUCLG1\tSUCLG2\tTDO2\tTP53INP2\tUBE2L6\tUGDH\tUROD\tUROS\tVNN1\tXIST\tYWHAH\nHALLMARK_G2M_CHECKPOINT\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_G2M_CHECKPOINT\tABL1\tAMD1\tARID4A\tATF5\tATRX\tAURKA\tAURKB\tBARD1\tBCL3\tBIRC5\tBRCA2\tBUB1\tBUB3\tCASP8AP2\tCBX1\tCCNA2\tCCNB2\tCCND1\tCCNF\tCCNT1\tCDC20\tCDC25A\tCDC25B\tCDC27\tCDC45\tCDC6\tCDC7\tCDK1\tCDK4\tCDKN1B\tCDKN2C\tCDKN3\tCENPA\tCENPE\tCENPF\tCHAF1A\tCHEK1\tCHMP1A\tCKS1B\tCKS2\tCTCF\tCUL1\tCUL3\tCUL4A\tCUL5\tDBF4\tDDX39A\tDKC1\tDMD\tDR1\tDTYMK\tE2F1\tE2F2\tE2F3\tE2F4\tEFNA5\tEGF\tESPL1\tEWSR1\tEXO1\tEZH2\tFANCC\tFBXO5\tFOXN3\tG3BP1\tGINS2\tGSPT1\tH2AX\tH2AZ1\tH2AZ2\tH2BC12\tHIF1A\tHIRA\tHMGA1\tHMGB3\tHMGN2\tHMMR\tHNRNPD\tHNRNPU\tHOXC10\tHSPA8\tHUS1\tILF3\tINCENP\tJPT1\tKATNA1\tKIF11\tKIF15\tKIF20B\tKIF22\tKIF23\tKIF2C\tKIF4A\tKIF5B\tKMT5A\tKNL1\tKPNA2\tKPNB1\tLBR\tLIG3\tLMNB1\tMAD2L1\tMAP3K20\tMAPK14\tMARCKS\tMCM2\tMCM3\tMCM5\tMCM6\tMEIS1\tMEIS2\tMKI67\tMNAT1\tMT2A\tMTF2\tMYBL2\tMYC\tNASP\tNCL\tNDC80\tNEK2\tNOLC1\tNOTCH2\tNSD2\tNUMA1\tNUP50\tNUP98\tNUSAP1\tODC1\tODF2\tORC5\tORC6\tPAFAH1B1\tPBK\tPDS5B\tPLK1\tPLK4\tPML\tPOLA2\tPOLE\tPOLQ\tPRC1\tPRIM2\tPRMT5\tPRP4K\tPTTG1\tPTTG3P\tPURA\tRACGAP1\tRAD21\tRAD23B\tRAD54L\tRASAL2\tRBL1\tRBM14\tRPA2\tRPS6KA5\tSAP30\tSFPQ\tSLC12A2\tSLC38A1\tSLC7A1\tSLC7A5\tSMAD3\tSMARCC1\tSMC1A\tSMC2\tSMC4\tSNRPD1\tSQLE\tSRSF1\tSRSF10\tSRSF2\tSS18\tSTAG1\tSTIL\tSTMN1\tSUV39H1\tSYNCRIP\tTACC3\tTENT4A\tTFDP1\tTGFB1\tTLE3\tTMPO\tTNPO2\tTOP1\tTOP2A\tTPX2\tTRA2B\tTRAIP\tTROAP\tTTK\tUBE2C\tUBE2S\tUCK2\tUPF1\tWRN\tXPO1\tYTHDC1\nHALLMARK_GLYCOLYSIS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_GLYCOLYSIS\tABCB6\tADORA2B\tAGL\tAGRN\tAK3\tAK4\tAKR1A1\tALDH7A1\tALDH9A1\tALDOA\tALDOB\tALG1\tANG\tANGPTL4\tANKZF1\tARPP19\tARTN\tAURKA\tB3GALT6\tB3GAT1\tB3GAT3\tB3GNT3\tB4GALT1\tB4GALT2\tB4GALT4\tB4GALT7\tBIK\tBPNT1\tCACNA1H\tCAPN5\tCASP6\tCD44\tCDK1\tCENPA\tCHPF\tCHPF2\tCHST1\tCHST12\tCHST2\tCHST4\tCHST6\tCITED2\tCLDN3\tCLDN9\tCLN6\tCOG2\tCOL5A1\tCOPB2\tCTH\tCXCR4\tCYB5A\tDCN\tDDIT4\tDEPDC1\tDLD\tDPYSL4\tDSC2\tECD\tEFNA3\tEGFR\tEGLN3\tELF3\tENO1\tENO2\tERO1A\tEXT1\tEXT2\tFAM162A\tFBP2\tFKBP4\tFUT8\tG6PD\tGAL3ST1\tGALE\tGALK1\tGALK2\tGAPDHS\tGCLC\tGFPT1\tGFUS\tGLCE\tGLRX\tGMPPA\tGMPPB\tGNE\tGNPDA1\tGOT1\tGOT2\tGPC1\tGPC3\tGPC4\tGPR87\tGUSB\tGYS1\tGYS2\tHAX1\tHDLBP\tHK2\tHMMR\tHOMER1\tHS2ST1\tHS6ST2\tHSPA5\tIDH1\tIDUA\tIER3\tIGFBP3\tIL13RA1\tIRS2\tISG20\tKDELR3\tKIF20A\tKIF2A\tLCT\tLDHA\tLDHC\tLHPP\tLHX9\tMDH1\tMDH2\tME1\tME2\tMED24\tMERTK\tMET\tMIF\tMIOX\tMPI\tMXI1\tNANP\tNASP\tNDST3\tNDUFV3\tNOL3\tNSDHL\tNT5E\tP4HA1\tP4HA2\tPAM\tPAXIP1\tPC\tPDK3\tPFKFB1\tPFKP\tPGAM1\tPGAM2\tPGK1\tPGLS\tPGM2\tPHKA2\tPKM\tPKP2\tPLOD1\tPLOD2\tPMM2\tPOLR3K\tPPFIA4\tPPIA\tPPP2CB\tPRPS1\tPSMC4\tPYGB\tPYGL\tQSOX1\tRARS1\tRBCK1\tRPE\tRRAGD\tSAP30\tSDC1\tSDC2\tSDC3\tSDHC\tSLC16A3\tSLC25A10\tSLC25A13\tSLC35A3\tSLC37A4\tSOD1\tSOX9\tSPAG4\tSRD5A3\tSTC1\tSTC2\tSTMN1\tTALDO1\tTFF3\tTGFA\tTGFBI\tTKTL1\tTPBG\tTPI1\tTPST1\tTXN\tUGP2\tVCAN\tVEGFA\tVLDLR\tXYLT2\tZNF292\nHALLMARK_HEDGEHOG_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_HEDGEHOG_SIGNALING\tACHE\tADGRG1\tAMOT\tCDK5R1\tCDK6\tCELSR1\tCNTFR\tCRMP1\tDPYSL2\tETS2\tGLI1\tHEY1\tHEY2\tL1CAM\tLDB1\tMYH9\tNF1\tNKX6-1\tNRCAM\tNRP1\tNRP2\tOPHN1\tPLG\tPML\tPTCH1\tRASA1\tRTN1\tSCG2\tSHH\tSLIT1\tTHY1\tTLE1\tTLE3\tUNC5C\tVEGFA\tVLDLR\nHALLMARK_HEME_METABOLISM\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_HEME_METABOLISM\tABCB6\tABCG2\tACKR1\tACP5\tACSL6\tADD1\tADD2\tADIPOR1\tAGPAT4\tAHSP\tALAD\tALAS2\tALDH1L1\tALDH6A1\tANK1\tAQP3\tARHGEF12\tARL2BP\tASNS\tATG4A\tATP6V0A1\tBACH1\tBCAM\tBLVRA\tBLVRB\tBMP2K\tBNIP3L\tBPGM\tBSG\tBTG2\tBTRC\tC3\tCA1\tCA2\tCAST\tCAT\tCCDC28A\tCCND3\tCDC27\tCDR2\tCIR1\tCLCN3\tCLIC2\tCPOX\tCROCCP2\tCTNS\tCTSB\tCTSE\tDAAM1\tDCAF10\tDCAF11\tDCUN1D1\tDMTN\tE2F2\tEIF2AK1\tELL2\tENDOD1\tEPB41\tEPB42\tEPOR\tERMAP\tEZH1\tFBXO34\tFBXO7\tFBXO9\tFECH\tFN3K\tFOXJ2\tFOXO3\tFTCD\tGAPVD1\tGATA1\tGCLC\tGCLM\tGDE1\tGLRX5\tGMPS\tGYPA\tGYPB\tGYPC\tGYPE\tH1-0\tH4C3\tHAGH\tHBB\tHBBP1\tHBD\tHBQ1\tHBZ\tHDGF\tHEBP1\tHMBS\tHTATIP2\tHTRA2\tICAM4\tIGSF3\tISCA1\tKAT2B\tKDM7A\tKEL\tKHNYN\tKLF1\tKLF3\tLAMP2\tLMO2\tLPIN2\tLRP10\tMAP2K3\tMARCHF2\tMARCHF8\tMARK3\tMBOAT2\tMFHAS1\tMGST3\tMINPP1\tMKRN1\tMOCOS\tMOSPD1\tMPP1\tMXI1\tMYL4\tNARF\tNCOA4\tNEK7\tNFE2\tNFE2L1\tNNT\tNR3C1\tNUDT4\tOPTN\tOSBP2\tP4HA2\tPC\tPDZK1IP1\tPGLS\tPICALM\tPIGQ\tPPOX\tPPP2R5B\tPRDX2\tPSMD9\tRAD23A\tRANBP10\tRAP1GAP\tRBM38\tRBM5\tRCL1\tRHAG\tRHCE\tRHD\tRIOK3\tRNF123\tRNF19A\tSDCBP\tSEC14L1\tSELENBP1\tSIDT2\tSLC10A3\tSLC11A2\tSLC22A4\tSLC25A37\tSLC25A38\tSLC2A1\tSLC30A1\tSLC30A10\tSLC4A1\tSLC66A2\tSLC6A8\tSLC6A9\tSLC7A11\tSMOX\tSNCA\tSPTA1\tSPTB\tSYNJ1\tTAL1\tTCEA1\tTENT5C\tTFDP2\tTFRC\tTMCC2\tTMEM9B\tTNRC6B\tTNS1\tTOP1\tTRAK2\tTRIM10\tTRIM58\tTSPAN5\tTSPO2\tTYR\tUBAC1\tUCP2\tUROD\tUROS\tUSP15\tVEZF1\tXK\tXPO7\tYPEL5\nHALLMARK_HYPOXIA\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_HYPOXIA\tACKR3\tADM\tADORA2B\tAK4\tAKAP12\tALDOA\tALDOB\tALDOC\tAMPD3\tANGPTL4\tANKZF1\tANXA2\tATF3\tATP7A\tB3GALT6\tB4GALNT2\tBCAN\tBCL2\tBGN\tBHLHE40\tBNIP3L\tBRS3\tBTG1\tCA12\tCASP6\tCAV1\tCAVIN1\tCAVIN3\tCCN1\tCCN2\tCCN5\tCCNG2\tCDKN1A\tCDKN1B\tCDKN1C\tCHST2\tCHST3\tCITED2\tCOL5A1\tCP\tCSRP2\tCXCR4\tDCN\tDDIT3\tDDIT4\tDPYSL4\tDTNA\tDUSP1\tEDN2\tEFNA1\tEFNA3\tEGFR\tENO1\tENO2\tENO3\tERO1A\tERRFI1\tETS1\tEXT1\tF3\tFAM162A\tFBP1\tFOS\tFOSL2\tFOXO3\tGAA\tGALK1\tGAPDH\tGAPDHS\tGBE1\tGCK\tGCNT2\tGLRX\tGPC1\tGPC3\tGPC4\tGPI\tGRHPR\tGYS1\tHAS1\tHDLBP\tHEXA\tHK1\tHK2\tHMOX1\tHOXB9\tHS3ST1\tHSPA5\tIDS\tIER3\tIGFBP1\tIGFBP3\tIL6\tILVBL\tINHA\tIRS2\tISG20\tJMJD6\tJUN\tKDELR3\tKDM3A\tKIF5A\tKLF6\tKLF7\tKLHL24\tLALBA\tLARGE1\tLDHA\tLDHC\tLOX\tLXN\tMAFF\tMAP3K1\tMIF\tMT1E\tMT2A\tMXI1\tMYH9\tNAGK\tNCAN\tNDRG1\tNDST1\tNDST2\tNEDD4L\tNFIL3\tNOCT\tNR3C1\tP4HA1\tP4HA2\tPAM\tPCK1\tPDGFB\tPDK1\tPDK3\tPFKFB3\tPFKL\tPFKP\tPGAM2\tPGF\tPGK1\tPGM1\tPGM2\tPHKG1\tPIM1\tPKLR\tPKP1\tPLAC8\tPLAUR\tPLIN2\tPNRC1\tPPARGC1A\tPPFIA4\tPPP1R15A\tPPP1R3C\tPRDX5\tPRKCA\tPYGM\tRBPJ\tRORA\tRRAGD\tS100A4\tSAP30\tSCARB1\tSDC2\tSDC3\tSDC4\tSELENBP1\tSERPINE1\tSIAH2\tSLC25A1\tSLC2A1\tSLC2A3\tSLC2A5\tSLC37A4\tSLC6A6\tSRPX\tSTBD1\tSTC1\tSTC2\tSULT2B1\tTES\tTGFB3\tTGFBI\tTGM2\tTIPARP\tTKTL1\tTMEM45A\tTNFAIP3\tTPBG\tTPD52\tTPI1\tTPST2\tUGP2\tVEGFA\tVHL\tVLDLR\tWSB1\tXPNPEP1\tZFP36\tZNF292\nHALLMARK_IL2_STAT5_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_IL2_STAT5_SIGNALING\tABCB1\tADAM19\tAGER\tAHCY\tAHNAK\tAHR\tALCAM\tAMACR\tANXA4\tAPLP1\tARL4A\tBATF\tBATF3\tBCL2\tBCL2L1\tBHLHE40\tBMP2\tBMPR2\tCA2\tCAPG\tCAPN3\tCASP3\tCCND2\tCCND3\tCCNE1\tCCR4\tCD44\tCD48\tCD79B\tCD81\tCD83\tCD86\tCDC42SE2\tCDC6\tCDCP1\tCDKN1C\tCISH\tCKAP4\tCOCH\tCOL6A1\tCSF1\tCSF2\tCST7\tCTLA4\tCTSZ\tCXCL10\tCYFIP1\tDCPS\tDENND5A\tDHRS3\tDRC1\tECM1\tEEF1AKMT1\tEMP1\tENO3\tENPP1\tEOMES\tETFBKMT\tETV4\tF2RL2\tFAH\tFGL2\tFLT3LG\tFURIN\tGABARAPL1\tGADD45B\tGALM\tGATA1\tGBP4\tGLIPR2\tGPR65\tGPR83\tGPX4\tGSTO1\tGUCY1B1\tHIPK2\tHK2\tHOPX\tHUWE1\tHYCC2\tICOS\tIFITM3\tIFNGR1\tIGF1R\tIGF2R\tIKZF2\tIKZF4\tIL10\tIL10RA\tIL13\tIL18R1\tIL1R2\tIL1RL1\tIL2RA\tIL2RB\tIL3RA\tIL4R\tIRF4\tIRF6\tIRF8\tITGA6\tITGAE\tITGAV\tITIH5\tKLF6\tLCLAT1\tLIF\tLRIG1\tLRRC8C\tLTB\tMAFF\tMAP3K8\tMAP6\tMAPKAPK2\tMUC1\tMXD1\tMYC\tMYO1C\tMYO1E\tNCOA3\tNCS1\tNDRG1\tNFIL3\tNFKBIZ\tNOP2\tNRP1\tNT5E\tODC1\tP2RX4\tP4HA1\tPDCD2L\tPENK\tPHLDA1\tPHTF2\tPIM1\tPLAGL1\tPLEC\tPLIN2\tPLPP1\tPLSCR1\tPNP\tPOU2F1\tPRAF2\tPRKCH\tPRNP\tPTCH1\tPTGER2\tPTH1R\tPTRH2\tPUS1\tRABGAP1L\tRGS16\tRHOB\tRHOH\tRNH1\tRORA\tRRAGD\tS100A1\tSCN9A\tSELL\tSELP\tSERPINB6\tSERPINC1\tSH3BGRL2\tSHE\tSLC1A5\tSLC29A2\tSLC2A3\tSLC39A8\tSMPDL3A\tSNX14\tSNX9\tSOCS1\tSOCS2\tSPP1\tSPRED2\tSPRY4\tST3GAL4\tSWAP70\tSYNGR2\tSYT11\tTGM2\tTIAM1\tTLR7\tTNFRSF18\tTNFRSF1B\tTNFRSF21\tTNFRSF4\tTNFRSF8\tTNFRSF9\tTNFSF10\tTNFSF11\tTRAF1\tTTC39B\tTWSG1\tUCK2\tUMPS\tWLS\tXBP1\nHALLMARK_IL6_JAK_STAT3_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_IL6_JAK_STAT3_SIGNALING\tA2M\tACVR1B\tACVRL1\tBAK1\tCBL\tCCL7\tCCR1\tCD14\tCD36\tCD38\tCD44\tCD9\tCNTFR\tCRLF2\tCSF1\tCSF2\tCSF2RA\tCSF2RB\tCSF3R\tCXCL1\tCXCL10\tCXCL11\tCXCL13\tCXCL3\tCXCL9\tDNTT\tEBI3\tFAS\tGRB2\tHAX1\tHMOX1\tIFNAR1\tIFNGR1\tIFNGR2\tIL10RB\tIL12RB1\tIL13RA1\tIL15RA\tIL17RA\tIL17RB\tIL18R1\tIL1B\tIL1R1\tIL1R2\tIL2RA\tIL2RG\tIL3RA\tIL4R\tIL6\tIL6ST\tIL7\tIL9R\tINHBE\tIRF1\tIRF9\tITGA4\tITGB3\tJUN\tLEPR\tLTB\tLTBR\tMAP3K8\tMYD88\tOSMR\tPDGFC\tPF4\tPIK3R5\tPIM1\tPLA2G2A\tPTPN1\tPTPN11\tPTPN2\tREG1A\tSOCS1\tSOCS3\tSTAM2\tSTAT1\tSTAT2\tSTAT3\tTGFB1\tTLR2\tTNF\tTNFRSF12A\tTNFRSF1A\tTNFRSF1B\tTNFRSF21\tTYK2\nHALLMARK_INFLAMMATORY_RESPONSE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_INFLAMMATORY_RESPONSE\tABCA1\tABI1\tACVR1B\tACVR2A\tADGRE1\tADM\tADORA2B\tADRM1\tAHR\tAPLNR\tAQP9\tATP2A2\tATP2B1\tATP2C1\tAXL\tBDKRB1\tBEST1\tBST2\tBTG2\tC3AR1\tC5AR1\tCALCRL\tCCL17\tCCL2\tCCL20\tCCL22\tCCL24\tCCL5\tCCL7\tCCR7\tCCRL2\tCD14\tCD40\tCD48\tCD55\tCD69\tCD70\tCD82\tCDKN1A\tCHST2\tCLEC5A\tCMKLR1\tCSF1\tCSF3\tCSF3R\tCX3CL1\tCXCL10\tCXCL11\tCXCL6\tCXCL8\tCXCL9\tCXCR6\tCYBB\tDCBLD2\tEBI3\tEDN1\tEIF2AK2\tEMP3\tEREG\tF3\tFFAR2\tFPR1\tFZD5\tGABBR1\tGCH1\tGNA15\tGNAI3\tGP1BA\tGPC3\tGPR132\tGPR183\tHAS2\tHBEGF\tHIF1A\tHPN\tHRH1\tICAM1\tICAM4\tICOSLG\tIFITM1\tIFNAR1\tIFNGR2\tIL10\tIL10RA\tIL12B\tIL15\tIL15RA\tIL18\tIL18R1\tIL18RAP\tIL1A\tIL1B\tIL1R1\tIL2RB\tIL4R\tIL6\tIL7R\tINHBA\tIRAK2\tIRF1\tIRF7\tITGA5\tITGB3\tITGB8\tKCNA3\tKCNJ2\tKCNMB2\tKIF1B\tKLF6\tLAMP3\tLCK\tLCP2\tLDLR\tLIF\tLPAR1\tLTA\tLY6E\tLYN\tMARCO\tMEFV\tMEP1A\tMET\tMMP14\tMSR1\tMXD1\tMYC\tNAMPT\tNDP\tNFKB1\tNFKBIA\tNLRP3\tNMI\tNMUR1\tNOD2\tNPFFR2\tOLR1\tOPRK1\tOSM\tOSMR\tP2RX4\tP2RX7\tP2RY2\tPCDH7\tPDE4B\tPDPN\tPIK3R5\tPLAUR\tPROK2\tPSEN1\tPTAFR\tPTGER2\tPTGER4\tPTGIR\tPTPRE\tPVR\tRAF1\tRASGRP1\tRELA\tRGS1\tRGS16\tRHOG\tRIPK2\tRNF144B\tROS1\tRTP4\tSCARF1\tSCN1B\tSELE\tSELENOS\tSELL\tSEMA4D\tSERPINE1\tSGMS2\tSLAMF1\tSLC11A2\tSLC1A2\tSLC28A2\tSLC31A1\tSLC31A2\tSLC4A4\tSLC7A1\tSLC7A2\tSPHK1\tSRI\tSTAB1\tTACR1\tTACR3\tTAPBP\tTIMP1\tTLR1\tTLR2\tTLR3\tTNFAIP6\tTNFRSF1B\tTNFRSF9\tTNFSF10\tTNFSF15\tTNFSF9\tTPBG\tVIP\nHALLMARK_INTERFERON_ALPHA_RESPONSE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_INTERFERON_ALPHA_RESPONSE\tADAR\tB2M\tBATF2\tBST2\tC1S\tCASP1\tCASP8\tCCRL2\tCD47\tCD74\tCMPK2\tCMTR1\tCNP\tCSF1\tCXCL10\tCXCL11\tDDX60\tDHX58\tEIF2AK2\tELF1\tEPSTI1\tGBP2\tGBP4\tGMPR\tHELZ2\tHERC6\tHLA-C\tIFI27\tIFI30\tIFI35\tIFI44\tIFI44L\tIFIH1\tIFIT2\tIFIT3\tIFITM1\tIFITM2\tIFITM3\tIL15\tIL4R\tIL7\tIRF1\tIRF2\tIRF7\tIRF9\tISG15\tISG20\tLAMP3\tLAP3\tLGALS3BP\tLPAR6\tLY6E\tMOV10\tMVB12A\tMX1\tNCOA7\tNMI\tNUB1\tOAS1\tOASL\tOGFR\tPARP12\tPARP14\tPARP9\tPLSCR1\tPNPT1\tPROCR\tPSMA3\tPSMB8\tPSMB9\tPSME1\tPSME2\tRIPK2\tRNF31\tRSAD2\tRTP4\tSAMD9\tSAMD9L\tSELL\tSLC25A28\tSP110\tSTAT2\tTAP1\tTDRD7\tTENT5A\tTMEM140\tTRAFD1\tTRIM14\tTRIM21\tTRIM25\tTRIM26\tTRIM5\tTXNIP\tUBA7\tUBE2L6\tUSP18\tWARS1\nHALLMARK_INTERFERON_GAMMA_RESPONSE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_INTERFERON_GAMMA_RESPONSE\tADAR\tAPOL6\tARID5B\tARL4A\tAUTS2\tB2M\tBANK1\tBATF2\tBPGM\tBST2\tBTG1\tC1R\tC1S\tCASP1\tCASP3\tCASP4\tCASP7\tCASP8\tCCL2\tCCL5\tCCL7\tCD274\tCD38\tCD40\tCD69\tCD74\tCD86\tCDKN1A\tCFB\tCFH\tCIITA\tCMKLR1\tCMPK2\tCMTR1\tCSF2RB\tCXCL10\tCXCL11\tCXCL9\tDDX60\tDHX58\tEIF2AK2\tEIF4E3\tEPSTI1\tFAS\tFCGR1A\tFGL2\tFPR1\tGBP4\tGBP6\tGCH1\tGPR18\tGZMA\tHELZ2\tHERC6\tHIF1A\tHLA-A\tHLA-B\tHLA-DMA\tHLA-DQA1\tHLA-DRB1\tHLA-G\tICAM1\tIDO1\tIFI27\tIFI30\tIFI35\tIFI44\tIFI44L\tIFIH1\tIFIT1\tIFIT2\tIFIT3\tIFITM2\tIFITM3\tIFNAR2\tIL10RA\tIL15\tIL15RA\tIL18BP\tIL2RB\tIL4R\tIL6\tIL7\tIRF1\tIRF2\tIRF4\tIRF5\tIRF7\tIRF8\tIRF9\tISG15\tISG20\tISOC1\tITGB7\tJAK2\tKLRK1\tLAP3\tLATS2\tLCP2\tLGALS3BP\tLY6E\tLYSMD2\tMARCHF1\tMT2A\tMTHFD2\tMVP\tMX1\tMX2\tMYD88\tNAMPT\tNCOA3\tNFKB1\tNFKBIA\tNLRC5\tNMI\tNOD1\tNUP93\tOAS2\tOAS3\tOASL\tOGFR\tP2RY14\tPARP12\tPARP14\tPDE4B\tPELI1\tPFKP\tPIM1\tPLA2G4A\tPLSCR1\tPML\tPNP\tPNPT1\tPSMA2\tPSMA3\tPSMB10\tPSMB2\tPSMB8\tPSMB9\tPSME1\tPSME2\tPTGS2\tPTPN1\tPTPN2\tPTPN6\tRAPGEF6\tRBCK1\tRIGI\tRIPK1\tRIPK2\tRNF213\tRNF31\tRSAD2\tRTP4\tSAMD9L\tSAMHD1\tSECTM1\tSELP\tSERPING1\tSLAMF7\tSLC25A28\tSOCS1\tSOCS3\tSOD2\tSP110\tSPPL2A\tSRI\tSSPN\tST3GAL5\tST8SIA4\tSTAT1\tSTAT2\tSTAT3\tSTAT4\tTAP1\tTAPBP\tTDRD7\tTMT1B\tTNFAIP2\tTNFAIP3\tTNFAIP6\tTNFSF10\tTOR1B\tTRAFD1\tTRIM14\tTRIM21\tTRIM25\tTRIM26\tTXNIP\tUBE2L6\tUPP1\tUSP18\tVAMP5\tVAMP8\tVCAM1\tWARS1\tXAF1\tXCL1\tZBP1\tZNFX1\nHALLMARK_KRAS_SIGNALING_DN\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_KRAS_SIGNALING_DN\tABCB11\tABCG4\tACTC1\tADRA2C\tAKR1B10\tALOX12B\tAMBN\tARHGDIG\tARPP21\tASB7\tATP4A\tATP6V1B1\tBARD1\tBMPR1B\tBRDT\tBTG2\tC5\tCACNA1F\tCACNG1\tCALCB\tCALML5\tCAMK1D\tCAPN9\tCCDC106\tCCNA1\tCCR8\tCD207\tCD40LG\tCD80\tCDH16\tCDKAL1\tCELSR2\tCHRNG\tCHST2\tCKM\tCLDN16\tCLDN8\tCLPS\tCLSTN3\tCNTFR\tCOL2A1\tCOPZ2\tCOQ8A\tCPA2\tCPB1\tCPEB3\tCYP11B2\tCYP39A1\tDCC\tDLK2\tDTNB\tEDAR\tEDN1\tEDN2\tEFHD1\tEGF\tENTPD7\tEPHA5\tFGF16\tFGF22\tFGFR3\tFGGY\tFSHB\tGAMT\tGDNF\tGP1BA\tGP2\tGPR19\tGPR3\tGPRC5C\tGRID2\tGTF3C5\tHNF1A\tHSD11B2\tHTR1B\tHTR1D\tIDUA\tIFI44L\tIFNG\tIGFBP2\tIL12B\tIL5\tINSL5\tIRS4\tITGB1BP2\tITIH3\tKCND1\tKCNE2\tKCNMB1\tKCNN1\tKCNQ2\tKLHDC8A\tKLK7\tKLK8\tKMT2D\tKRT1\tKRT13\tKRT15\tKRT4\tKRT5\tLFNG\tLGALS7\tLYPD3\tMACROH2A2\tMAGIX\tMAST3\tMEFV\tMFSD6\tMSH5\tMTHFR\tMX1\tMYH7\tMYO15A\tMYOT\tNGB\tNOS1\tNPHS1\tNPY4R\tNR4A2\tNR6A1\tNRIP2\tNTF3\tNUDT11\tOXT\tP2RX6\tP2RY4\tPAX3\tPAX4\tPCDHB1\tPDCD1\tPDE6B\tPDK2\tPKP1\tPLAG1\tPNMT\tPRKN\tPRODH\tPROP1\tPTGFR\tPTPRJ\tRGS11\tRIBC2\tRSAD2\tRYR1\tRYR2\tSCGB1A1\tSCN10A\tSELENOP\tSERPINA10\tSERPINB2\tSGK1\tSHOX2\tSIDT1\tSKIL\tSLC12A3\tSLC16A7\tSLC25A23\tSLC29A3\tSLC30A3\tSLC38A3\tSLC5A5\tSLC6A14\tSLC6A3\tSMPX\tSNCB\tSNN\tSOX10\tSPHK2\tSPRR3\tSPTBN2\tSSTR4\tSTAG3\tSYNPO\tTAS2R4\tTCF7L1\tTCL1A\tTENM2\tTENT5C\tTEX15\tTFAP2B\tTFCP2L1\tTFF2\tTG\tTGFB2\tTGM1\tTHNSL2\tTHRB\tTLX1\tTNNI3\tTSHB\tUGT2B17\tUPK3B\tVPREB1\tVPS50\tWNT16\tYBX2\tYPEL1\tZBTB16\tZC2HC1C\tZNF112\nHALLMARK_KRAS_SIGNALING_UP\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_KRAS_SIGNALING_UP\tABCB1\tACE\tADAM17\tADAM8\tADAMDEC1\tADGRA2\tADGRL4\tAKAP12\tAKT2\tALDH1A2\tALDH1A3\tAMMECR1\tANGPTL4\tANKH\tANO1\tANXA10\tAPOD\tARG1\tATG10\tAVL9\tBIRC3\tBMP2\tBPGM\tBTBD3\tBTC\tC3AR1\tCA2\tCAB39L\tCBL\tCBR4\tCBX8\tCCL20\tCCND2\tCCSER2\tCD37\tCDADC1\tCFB\tCFH\tCFHR2\tCIDEA\tCLEC4A\tCMKLR1\tCPE\tCROT\tCSF2\tCSF2RA\tCTSS\tCXCL10\tCXCR4\tDCBLD2\tDNMBP\tDOCK2\tDUSP6\tEMP1\tENG\tEPB41L3\tEPHB2\tEREG\tERO1A\tETS1\tETV1\tETV4\tETV5\tEVI5\tF13A1\tF2RL1\tFBXO4\tFCER1G\tFGF9\tFLT4\tFUCA1\tG0S2\tGABRA3\tGADD45G\tGALNT3\tGFPT2\tGLRX\tGNG11\tGPNMB\tGPRC5B\tGUCY1A1\tGYPC\tH2BC3\tHBEGF\tHDAC9\tHKDC1\tHOXD11\tHSD11B1\tID2\tIGF2\tIGFBP3\tIKZF1\tIL10RA\tIL1B\tIL1RL2\tIL2RG\tIL33\tIL7R\tINHBA\tIRF8\tITGA2\tITGB2\tITGBL1\tJUP\tKCNN4\tKIF5C\tKLF4\tLAPTM5\tLAT2\tLCP1\tLIF\tLY96\tMAFB\tMALL\tMAP3K1\tMAP4K1\tMAP7\tMMD\tMMP10\tMMP11\tMMP9\tMPZL2\tMTMR10\tMYCN\tNAP1L2\tNGF\tNIN\tNR0B2\tNR1H4\tNRP1\tPCP4\tPCSK1N\tPDCD1LG2\tPECAM1\tPEG3\tPIGR\tPLAT\tPLAU\tPLAUR\tPLEK2\tPLVAP\tPPBP\tPPP1R15A\tPRDM1\tPRELID3B\tPRKG2\tPRRX1\tPSMB8\tPTBP2\tPTCD2\tPTGS2\tPTPRR\tRABGAP1L\tRBM4\tRBP4\tRELN\tRETN\tRGS16\tSATB1\tSCG3\tSCG5\tSCN1B\tSDCCAG8\tSEMA3B\tSERPINA3\tSLPI\tSNAP25\tSNAP91\tSOX9\tSPARCL1\tSPON1\tSPP1\tSPRY2\tST6GAL1\tSTRN\tTFPI\tTLR8\tTMEM100\tTMEM158\tTMEM176A\tTMEM176B\tTNFAIP3\tTNFRSF1B\tTNNT2\tTOR1AIP2\tTPH1\tTRAF1\tTRIB1\tTRIB2\tTSPAN1\tTSPAN13\tTSPAN7\tUSH1C\tUSP12\tVWA5A\tWDR33\tWNT7A\tYRDC\tZNF277\tZNF639\nHALLMARK_MITOTIC_SPINDLE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_MITOTIC_SPINDLE\tABI1\tABL1\tABR\tACTN4\tAKAP13\tALMS1\tALS2\tANLN\tAPC\tARAP3\tARF6\tARFGEF1\tARFIP2\tARHGAP10\tARHGAP27\tARHGAP29\tARHGAP4\tARHGAP5\tARHGDIA\tARHGEF11\tARHGEF12\tARHGEF2\tARHGEF3\tARHGEF7\tARL8A\tATG4B\tAURKA\tBCAR1\tBCL2L11\tBCR\tBIN1\tBIRC5\tBRCA2\tBUB1\tCAPZB\tCCDC88A\tCCNB2\tCD2AP\tCDC27\tCDC42\tCDC42BPA\tCDC42EP1\tCDC42EP2\tCDC42EP4\tCDK1\tCDK5RAP2\tCENPE\tCENPF\tCEP131\tCEP192\tCEP250\tCEP57\tCEP72\tCKAP5\tCLASP1\tCLIP1\tCLIP2\tCNTRL\tCNTROB\tCPAP\tCSNK1D\tCTTN\tCYTH2\tDLG1\tDLGAP5\tDOCK2\tDOCK4\tDST\tDYNC1H1\tDYNLL2\tECT2\tEPB41\tEPB41L2\tESPL1\tEZR\tFARP1\tFBXO5\tFGD4\tFGD6\tFLNA\tFLNB\tFSCN1\tGEMIN4\tGSN\tHDAC6\tHOOK3\tINCENP\tITSN1\tKATNA1\tKATNB1\tKIF11\tKIF15\tKIF1B\tKIF20B\tKIF22\tKIF23\tKIF2C\tKIF3B\tKIF3C\tKIF4A\tKIF5B\tKIFAP3\tKLC1\tKNTC1\tKPTN\tLATS1\tLLGL1\tLMNB1\tLRPPRC\tMAP1S\tMAP3K11\tMAPRE1\tMARCKS\tMARK4\tMID1\tMID1IP1\tMYH10\tMYH9\tMYO1E\tMYO9B\tNCK1\tNCK2\tNDC80\tNEDD9\tNEK2\tNET1\tNF1\tNIN\tNOTCH2\tNUMA1\tNUSAP1\tOPHN1\tPAFAH1B1\tPALLD\tPCGF5\tPCM1\tPCNT\tPDLIM5\tPIF1\tPKD2\tPLEKHG2\tPLK1\tPPP4R2\tPRC1\tPREX1\tPXN\tRAB3GAP1\tRABGAP1\tRACGAP1\tRALBP1\tRANBP9\tRAPGEF5\tRAPGEF6\tRASA1\tRASA2\tRASAL2\tRFC1\tRHOF\tRHOT2\tRICTOR\tROCK1\tSAC3D1\tSASS6\tSEPTIN9\tSHROOM1\tSHROOM2\tSMC1A\tSMC3\tSMC4\tSORBS2\tSOS1\tSPTAN1\tSPTBN1\tSSH2\tSTAU1\tSTK38L\tSUN2\tSYNPO\tTAOK2\tTBCD\tTIAM1\tTLK1\tTOP2A\tTPX2\tTRIO\tTSC1\tTTK\tTUBA4A\tTUBD1\tTUBGCP2\tTUBGCP3\tTUBGCP5\tTUBGCP6\tUXT\tVCL\tWASF1\tWASF2\tWASL\tYWHAE\nHALLMARK_MTORC1_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_MTORC1_SIGNALING\tABCF2\tACACA\tACLY\tACSL3\tACTR2\tACTR3\tADD3\tADIPOR2\tAK4\tALDOA\tARPC5L\tASNS\tATP2A2\tATP5MC1\tATP6V1D\tAURKA\tBCAT1\tBHLHE40\tBTG2\tBUB1\tCACYBP\tCALR\tCANX\tCCNF\tCCNG1\tCCT6A\tCD9\tCDC25A\tCDKN1A\tCFP\tCOPS5\tCORO1A\tCTH\tCTSC\tCXCR4\tCYB5B\tCYP51A1\tDAPP1\tDDIT3\tDDIT4\tDDX39A\tDHCR24\tDHCR7\tDHFR\tEBP\tEDEM1\tEEF1E1\tEGLN3\tEIF2S2\tELOVL5\tELOVL6\tENO1\tEPRS1\tERO1A\tETF1\tFADS1\tFADS2\tFDXR\tFGL2\tFKBP2\tG6PD\tGAPDH\tGBE1\tGCLC\tGGA2\tGLA\tGLRX\tGMPS\tGOT1\tGPI\tGSK3B\tGSR\tGTF2H1\tHK2\tHMBS\tHMGCR\tHMGCS1\tHPRT1\tHSP90B1\tHSPA4\tHSPA5\tHSPA9\tHSPD1\tHSPE1\tIDH1\tIDI1\tIFI30\tIFRD1\tIGFBP5\tIMMT\tINSIG1\tITGB2\tLDHA\tLDLR\tLGMN\tLTA4H\tM6PR\tMAP2K3\tMCM2\tMCM4\tME1\tMLLT11\tMTHFD2\tMTHFD2L\tNAMPT\tNFIL3\tNFKBIB\tNFYC\tNHERF1\tNIBAN1\tNMT1\tNUFIP1\tNUP205\tNUPR1\tP4HA1\tPDAP1\tPDK1\tPFKL\tPGK1\tPGM1\tPHGDH\tPIK3R3\tPITPNB\tPLK1\tPLOD2\tPNO1\tPNP\tPOLR3G\tPPA1\tPPIA\tPPP1R15A\tPRDX1\tPSAT1\tPSMA3\tPSMA4\tPSMB5\tPSMC2\tPSMC4\tPSMC6\tPSMD12\tPSMD13\tPSMD14\tPSME3\tPSMG1\tPSPH\tQDPR\tRAB1A\tRDH11\tRIT1\tRPA1\tRPN1\tRRM2\tRRP9\tSC5D\tSCD\tSDF2L1\tSEC11A\tSERP1\tSERPINH1\tSHMT2\tSKAP2\tSLA\tSLC1A4\tSLC1A5\tSLC2A1\tSLC2A3\tSLC37A4\tSLC6A6\tSLC7A11\tSLC7A5\tSORD\tSQLE\tSQSTM1\tSRD5A1\tSSR1\tSTARD4\tSTC1\tSTIP1\tSYTL2\tTBK1\tTCEA1\tTES\tTFRC\tTM7SF2\tTMEM97\tTOMM40\tTPI1\tTRIB3\tTUBA4A\tTUBG1\tTXNRD1\tUBE2D3\tUCHL5\tUFM1\tUNG\tUSO1\tVLDLR\tWARS1\tXBP1\tYKT6\nHALLMARK_MYC_TARGETS_V1\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_MYC_TARGETS_V1\tABCE1\tACP1\tAIMP2\tAP3S1\tAPEX1\tBUB3\tC1QBP\tCAD\tCANX\tCBX3\tCCNA2\tCCT2\tCCT3\tCCT4\tCCT5\tCCT7\tCDC20\tCDC45\tCDK2\tCDK4\tCLNS1A\tCNBP\tCOPS5\tCOX5A\tCSTF2\tCTPS1\tCUL1\tCYC1\tDDX18\tDDX21\tDEK\tDHX15\tDUT\tEEF1B2\tEIF1AX\tEIF2S1\tEIF2S2\tEIF3B\tEIF3D\tEIF3J\tEIF4A1\tEIF4E\tEIF4G2\tEIF4H\tEPRS1\tERH\tETF1\tEXOSC7\tFAM120A\tFBL\tG3BP1\tGLO1\tGNL3\tGOT2\tGSPT1\tH2AZ1\tHDAC2\tHDDC2\tHDGF\tHNRNPA1\tHNRNPA2B1\tHNRNPA3\tHNRNPC\tHNRNPD\tHNRNPR\tHNRNPU\tHPRT1\tHSP90AB1\tHSPD1\tHSPE1\tIARS1\tIFRD1\tILF2\tIMPDH2\tKARS1\tKPNA2\tKPNB1\tLDHA\tLSM2\tLSM7\tMAD2L1\tMCM2\tMCM4\tMCM5\tMCM6\tMCM7\tMRPL23\tMRPL9\tMRPS18B\tMYC\tNAP1L1\tNCBP1\tNCBP2\tNDUFAB1\tNHP2\tNME1\tNOLC1\tNOP16\tNOP56\tNPM1\tODC1\tORC2\tPA2G4\tPABPC1\tPABPC4\tPCBP1\tPCNA\tPGK1\tPHB1\tPHB2\tPOLD2\tPOLE3\tPPIA\tPPM1G\tPRDX3\tPRDX4\tPRPF31\tPRPS2\tPSMA1\tPSMA2\tPSMA4\tPSMA6\tPSMA7\tPSMB2\tPSMB3\tPSMC4\tPSMC6\tPSMD1\tPSMD14\tPSMD3\tPSMD7\tPSMD8\tPTGES3\tPWP1\tRACK1\tRAD23B\tRAN\tRANBP1\tRFC4\tRNPS1\tRPL14\tRPL18\tRPL22\tRPL34\tRPL6\tRPLP0\tRPS10\tRPS2\tRPS3\tRPS5\tRPS6\tRRM1\tRRP9\tRSL1D1\tRUVBL2\tSERBP1\tSET\tSF3A1\tSF3B3\tSLC25A3\tSMARCC1\tSNRPA\tSNRPA1\tSNRPB2\tSNRPD1\tSNRPD2\tSNRPD3\tSNRPG\tSRM\tSRPK1\tSRSF1\tSRSF2\tSRSF3\tSRSF7\tSSB\tSSBP1\tSTARD7\tSYNCRIP\tTARDBP\tTCP1\tTFDP1\tTOMM70\tTRA2B\tTRIM28\tTUFM\tTXNL4A\tTYMS\tU2AF1\tUBA2\tUBE2E1\tUBE2L3\tUSP1\tVBP1\tVDAC1\tVDAC3\tXPO1\tXPOT\tXRCC6\tYWHAE\tYWHAQ\nHALLMARK_MYC_TARGETS_V2\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_MYC_TARGETS_V2\tAIMP2\tBYSL\tCBX3\tCDK4\tDCTPP1\tDDX18\tDUSP2\tEXOSC5\tFARSA\tGNL3\tGRWD1\tHK2\tHSPD1\tHSPE1\tIMP4\tIPO4\tLAS1L\tMAP3K6\tMCM4\tMCM5\tMPHOSPH10\tMRTO4\tMYBBP1A\tMYC\tNDUFAF4\tNIP7\tNOC4L\tNOLC1\tNOP16\tNOP2\tNOP56\tNPM1\tPA2G4\tPES1\tPHB1\tPLK1\tPLK4\tPPAN\tPPRC1\tPRMT3\tPUS1\tRABEPK\tRCL1\tRRP12\tRRP9\tSLC19A1\tSLC29A2\tSORD\tSRM\tSUPV3L1\tTBRG4\tTCOF1\tTFB2M\tTMEM97\tUNG\tUTP20\tWDR43\tWDR74\nHALLMARK_MYOGENESIS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_MYOGENESIS\tABLIM1\tACHE\tACSL1\tACTA1\tACTC1\tACTN2\tACTN3\tADAM12\tADCY9\tAEBP1\tAGL\tAGRN\tAK1\tAKT2\tANKRD2\tAPLNR\tAPOD\tAPP\tATP2A1\tATP6AP1\tBAG1\tBDKRB2\tBHLHE40\tBIN1\tCACNA1H\tCACNG1\tCAMK2B\tCASQ1\tCASQ2\tCAV3\tCD36\tCDH13\tCDKN1A\tCFD\tCHRNA1\tCHRNB1\tCHRNG\tCKB\tCKM\tCKMT2\tCLU\tCNN3\tCOL15A1\tCOL1A1\tCOL3A1\tCOL4A2\tCOL6A2\tCOL6A3\tCOX6A2\tCOX7A1\tCRAT\tCRYAB\tCSRP3\tCTF1\tDAPK2\tDENND2B\tDES\tDMD\tDMPK\tDTNA\tEFS\tEIF4A2\tENO3\tEPHB3\tERBB3\tFABP3\tFDPS\tFGF2\tFHL1\tFKBP1B\tFLII\tFOXO4\tFST\tFXYD1\tGAA\tGABARAPL2\tGADD45B\tGJA5\tGNAO1\tGPX3\tGSN\tHBEGF\tHDAC5\tHRC\tHSPB2\tHSPB8\tIFRD1\tIGF1\tIGFBP3\tIGFBP7\tITGA7\tITGB1\tITGB4\tITGB5\tKCNH1\tKCNH2\tKIFC3\tKLF5\tLAMA2\tLARGE1\tLDB3\tLPIN1\tLSP1\tMAPK12\tMAPRE3\tMB\tMEF2A\tMEF2C\tMEF2D\tMRAS\tMYBPC3\tMYBPH\tMYF6\tMYH1\tMYH11\tMYH2\tMYH3\tMYH4\tMYH7\tMYH8\tMYH9\tMYL1\tMYL11\tMYL2\tMYL3\tMYL4\tMYL6B\tMYL7\tMYLK\tMYO1C\tMYOG\tMYOM1\tMYOM2\tMYOZ1\tNAV2\tNCAM1\tNOS1\tNOTCH1\tNQO1\tOCEL1\tPC\tPDE4DIP\tPDLIM7\tPFKM\tPGAM2\tPICK1\tPKIA\tPLXNB2\tPPFIA4\tPPP1R3C\tPRNP\tPSEN2\tPTGIS\tPTP4A3\tPVALB\tPYGM\tRB1\tREEP1\tRIT1\tRYR1\tSCD\tSCHIP1\tSGCA\tSGCD\tSGCG\tSH2B1\tSH3BGR\tSIRT2\tSLC6A8\tSLN\tSMTN\tSOD3\tSORBS1\tSORBS3\tSPARC\tSPDEF\tSPEG\tSPHK1\tSPTAN1\tSSPN\tSTC2\tSVIL\tSYNGR2\tTAGLN\tTCAP\tTEAD4\tTGFB1\tTNNC1\tTNNC2\tTNNI1\tTNNI2\tTNNT1\tTNNT2\tTNNT3\tTPD52L1\tTPM2\tTPM3\tTSC2\tVIPR1\tWWTR1\nHALLMARK_NOTCH_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_NOTCH_SIGNALING\tAPH1A\tARRB1\tCCND1\tCUL1\tDLL1\tDTX1\tDTX2\tDTX4\tFBXW11\tFZD1\tFZD5\tFZD7\tHES1\tHEYL\tJAG1\tKAT2A\tLFNG\tMAML2\tNOTCH1\tNOTCH2\tNOTCH3\tPPARD\tPRKCA\tPSEN2\tPSENEN\tRBX1\tSAP30\tSKP1\tST3GAL6\tTCF7L2\tWNT2\tWNT5A\nHALLMARK_OXIDATIVE_PHOSPHORYLATION\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_OXIDATIVE_PHOSPHORYLATION\tABCB7\tACAA1\tACAA2\tACADM\tACADSB\tACADVL\tACAT1\tACO2\tAFG3L2\tAIFM1\tALAS1\tALDH6A1\tATP1B1\tATP5F1A\tATP5F1B\tATP5F1C\tATP5F1D\tATP5F1E\tATP5MC1\tATP5MC2\tATP5MC3\tATP5ME\tATP5MF\tATP5MG\tATP5PB\tATP5PD\tATP5PF\tATP5PO\tATP6AP1\tATP6V0B\tATP6V0C\tATP6V0E1\tATP6V1C1\tATP6V1D\tATP6V1E1\tATP6V1F\tATP6V1G1\tATP6V1H\tBAX\tBCKDHA\tBDH2\tCASP7\tCOX10\tCOX11\tCOX15\tCOX17\tCOX4I1\tCOX5A\tCOX5B\tCOX6A1\tCOX6B1\tCOX6C\tCOX7A2\tCOX7A2L\tCOX7B\tCOX7C\tCOX8A\tCPT1A\tCS\tCYB5A\tCYB5R3\tCYC1\tCYCS\tDECR1\tDLAT\tDLD\tDLST\tECH1\tECHS1\tECI1\tETFA\tETFB\tETFDH\tFDX1\tFH\tFXN\tGLUD1\tGOT2\tGPI\tGPX4\tGRPEL1\tHADHA\tHADHB\tHCCS\tHSD17B10\tHSPA9\tHTRA2\tIDH1\tIDH2\tIDH3A\tIDH3B\tIDH3G\tIMMT\tISCA1\tISCU\tLDHA\tLDHB\tLRPPRC\tMAOB\tMDH1\tMDH2\tMFN2\tMGST3\tMPC1\tMRPL11\tMRPL15\tMRPL34\tMRPL35\tMRPS11\tMRPS12\tMRPS15\tMRPS22\tMRPS30\tMTRF1\tMTRR\tMTX2\tNDUFA1\tNDUFA2\tNDUFA3\tNDUFA4\tNDUFA5\tNDUFA6\tNDUFA7\tNDUFA8\tNDUFA9\tNDUFAB1\tNDUFB1\tNDUFB2\tNDUFB3\tNDUFB4\tNDUFB5\tNDUFB6\tNDUFB7\tNDUFB8\tNDUFC1\tNDUFC2\tNDUFS1\tNDUFS2\tNDUFS3\tNDUFS4\tNDUFS6\tNDUFS7\tNDUFS8\tNDUFV1\tNDUFV2\tNNT\tNQO2\tOAT\tOGDH\tOPA1\tOXA1L\tPDHA1\tPDHB\tPDHX\tPDK4\tPDP1\tPHB2\tPHYH\tPMPCA\tPOLR2F\tPOR\tPRDX3\tRETSAT\tRHOT1\tRHOT2\tSDHA\tSDHB\tSDHC\tSDHD\tSLC25A11\tSLC25A12\tSLC25A20\tSLC25A3\tSLC25A4\tSLC25A5\tSLC25A6\tSUCLA2\tSUCLG1\tSUPV3L1\tSURF1\tTCIRG1\tTIMM10\tTIMM13\tTIMM17A\tTIMM50\tTIMM8B\tTIMM9\tTOMM22\tTOMM70\tUQCR10\tUQCR11\tUQCRB\tUQCRC1\tUQCRC2\tUQCRFS1\tUQCRH\tUQCRQ\tVDAC1\tVDAC2\tVDAC3\nHALLMARK_P53_PATHWAY\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_P53_PATHWAY\tABAT\tABCC5\tABHD4\tACVR1B\tADA\tAEN\tAK1\tALOX15B\tANKRA2\tAPAF1\tAPP\tATF3\tBAIAP2\tBAK1\tBAX\tBLCAP\tBMP2\tBTG1\tBTG2\tCASP1\tCCND2\tCCND3\tCCNG1\tCCNK\tCCP110\tCD81\tCD82\tCDH13\tCDK5R1\tCDKN1A\tCDKN2A\tCDKN2AIP\tCDKN2B\tCEBPA\tCGRRF1\tCLCA2\tCOQ8A\tCSRNP2\tCTSD\tCTSF\tCYFIP2\tDCXR\tDDB2\tDDIT3\tDDIT4\tDEF6\tDGKA\tDNTTIP2\tDRAM1\tEI24\tELP1\tEPHA2\tEPHX1\tEPS8L2\tERCC5\tF2R\tFAM162A\tFAS\tFBXW7\tFDXR\tFGF13\tFOS\tFOXO3\tFUCA1\tGADD45A\tGLS2\tGM2A\tGPX2\tH1-2\tH2AC25\tH2AJ\tHBEGF\tHDAC3\tHEXIM1\tHINT1\tHMOX1\tHRAS\tHSPA4L\tIER3\tIER5\tIFI30\tIL1A\tINHBB\tIP6K2\tIRAG2\tIRAK1\tISCU\tITGB4\tJAG2\tJUN\tKIF13B\tKLF4\tKLK8\tKRT17\tLDHB\tLIF\tMAPKAPK3\tMDM2\tMKNK2\tMXD1\tMXD4\tNDRG1\tNHLH2\tNINJ1\tNOL8\tNOTCH1\tNUDT15\tNUPR1\tOSGIN1\tPCNA\tPDGFA\tPERP\tPHLDA3\tPIDD1\tPITPNC1\tPLK2\tPLK3\tPLXNB2\tPMM1\tPOLH\tPOM121\tPPM1D\tPPP1R15A\tPRKAB1\tPRMT2\tPROCR\tPTPN14\tPTPRE\tPVT1\tRAB40C\tRACK1\tRAD51C\tRAD9A\tRALGDS\tRAP2B\tRB1\tRCHY1\tRETSAT\tRGS16\tRHBDF2\tRNF19B\tRPL18\tRPL36\tRPS12\tRPS27L\tRRAD\tRRP8\tRXRA\tS100A10\tS100A4\tSAT1\tSDC1\tSEC61A1\tSERPINB5\tSERTAD3\tSESN1\tSFN\tSLC19A2\tSLC35D1\tSLC3A2\tSLC7A11\tSOCS1\tSP1\tSPHK1\tST14\tSTEAP3\tSTOM\tTAP1\tTAX1BP3\tTCHH\tTCN2\tTGFA\tTGFB1\tTM4SF1\tTM7SF3\tTNFSF9\tTNNI1\tTOB1\tTP53\tTP63\tTPD52L1\tTPRKB\tTRAF4\tTRAFD1\tTRIAP1\tTRIB3\tTSC22D1\tTSPYL2\tTXNIP\tUPP1\tVAMP8\tVDR\tVWA5A\tWRAP73\tWWP1\tXPC\tZBTB16\tZFP36L1\tZMAT3\tZNF365\nHALLMARK_PANCREAS_BETA_CELLS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_PANCREAS_BETA_CELLS\tABCC8\tAKT3\tCHGA\tDCX\tDPP4\tELP4\tFOXA2\tFOXO1\tG6PC2\tGCG\tGCK\tHNF1A\tIAPP\tINS\tINSM1\tISL1\tLMO2\tMAFB\tNEUROD1\tNEUROG3\tNKX2-2\tNKX6-1\tPAK3\tPAX4\tPAX6\tPCSK1\tPCSK2\tPDX1\tPKLR\tSCGN\tSEC11A\tSLC2A2\tSPCS1\tSRP14\tSRP9\tSRPRB\tSST\tSTXBP1\tSYT13\tVDR\nHALLMARK_PEROXISOME\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_PEROXISOME\tABCB1\tABCB4\tABCB9\tABCC5\tABCC8\tABCD1\tABCD2\tABCD3\tACAA1\tACOT8\tACOX1\tACSL1\tACSL4\tACSL5\tALB\tALDH1A1\tALDH9A1\tATXN1\tBCL10\tCACNA1B\tCADM1\tCAT\tCDK7\tCEL\tCLN6\tCLN8\tCNBP\tCRABP1\tCRABP2\tCRAT\tCTBP1\tCTPS1\tDHCR24\tDHRS3\tDIO1\tDLG4\tECH1\tECI2\tEHHADH\tELOVL5\tEPHX2\tERCC1\tERCC3\tESR2\tFABP6\tFADS1\tFDPS\tFIS1\tGNPAT\tGSTK1\tHAO2\tHMGCL\tHRAS\tHSD11B2\tHSD17B11\tHSD17B4\tHSD3B7\tIDE\tIDH1\tIDH2\tIDI1\tISOC1\tITGB1BP1\tLONP2\tMLYCD\tMSH2\tMVP\tNR1I2\tNUDT19\tPABPC1\tPEX11A\tPEX11B\tPEX13\tPEX14\tPEX2\tPEX5\tPEX6\tPRDX1\tPRDX5\tRDH11\tRETSAT\tRXRG\tSCGB1A1\tSCP2\tSEMA3C\tSERPINA6\tSIAH1\tSLC23A2\tSLC25A17\tSLC25A19\tSLC25A4\tSLC27A2\tSLC35B2\tSMARCC1\tSOD1\tSOD2\tSTS\tSULT2B1\tTOP2A\tTSPO\tTTR\tUGT2B17\tVPS4B\tYWHAH\nHALLMARK_PI3K_AKT_MTOR_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_PI3K_AKT_MTOR_SIGNALING\tACACA\tACTR2\tACTR3\tADCY2\tAKT1\tAKT1S1\tAP2M1\tARF1\tARHGDIA\tARPC3\tATF1\tCAB39\tCAB39L\tCALR\tCAMK4\tCDK1\tCDK2\tCDK4\tCDKN1A\tCDKN1B\tCFL1\tCLTC\tCSNK2B\tCXCR4\tDAPP1\tDDIT3\tDUSP3\tE2F1\tECSIT\tEGFR\tEIF4E\tFASLG\tFGF17\tFGF22\tFGF6\tGNA14\tGNGT1\tGRB2\tGRK2\tGSK3B\tHRAS\tHSP90B1\tIL2RG\tIL4\tIRAK4\tITPR2\tLCK\tMAP2K3\tMAP2K6\tMAP3K7\tMAPK1\tMAPK10\tMAPK8\tMAPK9\tMAPKAP1\tMKNK1\tMKNK2\tMYD88\tNCK1\tNFKBIB\tNGF\tNOD1\tPAK4\tPDK1\tPFN1\tPIK3R3\tPIKFYVE\tPIN1\tPITX2\tPLA2G12A\tPLCB1\tPLCG1\tPPP1CA\tPPP2R1B\tPRKAA2\tPRKAG1\tPRKAR2A\tPRKCB\tPTEN\tPTPN11\tRAC1\tRAF1\tRALB\tRIPK1\tRIT1\tRPS6KA1\tRPS6KA3\tRPTOR\tSFN\tSLA\tSLC2A1\tSMAD2\tSQSTM1\tSTAT2\tTBK1\tTHEM4\tTIAM1\tTNFRSF1A\tTRAF2\tTRIB3\tTSC2\tUBE2D3\tUBE2N\tVAV3\tYWHAB\nHALLMARK_PROTEIN_SECRETION\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_PROTEIN_SECRETION\tABCA1\tADAM10\tANP32E\tAP1G1\tAP2B1\tAP2M1\tAP2S1\tAP3B1\tAP3S1\tARCN1\tARF1\tARFGAP3\tARFGEF1\tARFGEF2\tARFIP1\tATP1A1\tATP6V1B1\tATP6V1H\tATP7A\tBET1\tBNIP3\tCAV2\tCD63\tCLCN3\tCLN5\tCLTA\tCLTC\tCOG2\tCOPB1\tCOPB2\tCOPE\tCTSC\tDNM1L\tDOP1A\tDST\tEGFR\tERGIC3\tGALC\tGBF1\tGLA\tGNAS\tGOLGA4\tGOSR2\tICA1\tIGF2R\tKIF1B\tKRT18\tLAMP2\tLMAN1\tM6PR\tMAPK1\tMON2\tNAPA\tNAPG\tOCRL\tPAM\tPPT1\tRAB14\tRAB22A\tRAB2A\tRAB5A\tRAB9A\tRER1\tRPS6KA3\tSCAMP1\tSCAMP3\tSCRN1\tSEC22B\tSEC24D\tSEC31A\tSGMS1\tSH3GL2\tSNAP23\tSNX2\tSOD1\tSSPN\tSTAM\tSTX12\tSTX16\tSTX7\tTMED10\tTMED2\tTMX1\tTOM1L1\tTPD52\tTSG101\tTSPAN8\tUSO1\tVAMP3\tVAMP4\tVAMP7\tVPS45\tVPS4B\tYIPF6\tYKT6\tZW10\nHALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY\tABCC1\tATOX1\tCAT\tCDKN2D\tEGLN2\tERCC2\tFES\tFTL\tG6PD\tGCLC\tGCLM\tGLRX\tGLRX2\tGPX3\tGPX4\tGSR\tHHEX\tHMOX2\tIPCEF1\tJUNB\tLAMTOR5\tLSP1\tMBP\tMGST1\tMPO\tMSRA\tNDUFA6\tNDUFB4\tNDUFS2\tNQO1\tOXSR1\tPDLIM1\tPFKP\tPRDX1\tPRDX2\tPRDX4\tPRDX6\tPRNP\tPTPA\tSBNO2\tSCAF4\tSELENOS\tSOD1\tSOD2\tSRXN1\tSTK25\tTXN\tTXNRD1\tTXNRD2\nHALLMARK_SPERMATOGENESIS\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_SPERMATOGENESIS\tACE\tACRBP\tACRV1\tACTL7B\tADAD1\tADAM2\tADCYAP1\tAGFG1\tAKAP4\tALOX15\tARL4A\tART3\tAURKA\tBRAF\tBUB1\tCAMK4\tCCNA1\tCCNB2\tCCT6B\tCDK1\tCDKN3\tCFTR\tCHFR\tCHRM4\tCLGN\tCLPB\tCLVS1\tCNIH2\tCOIL\tCRISP2\tCSNK2A2\tCST8\tDBF4\tDCC\tDDX25\tDDX4\tDMC1\tDMRT1\tDNAJB8\tDPEP3\tELOVL3\tEZH2\tGAD1\tGAPDHS\tGFI1\tGMCL1\tGPR182\tGRM8\tGSG1\tGSTM3\tH1-6\tHBZ\tHOXB1\tHSPA1L\tHSPA2\tHSPA4L\tHTR5A\tIDE\tIFT88\tIL12RB2\tIL13RA2\tIP6K1\tJAM3\tKIF2C\tLDHC\tLPIN1\tMAP7\tMAST2\tMEP1B\tMLF1\tMLLT10\tMTNR1A\tMTOR\tNAA11\tNCAPH\tNEFH\tNEK2\tNF2\tNOS1\tNPHP1\tNPY5R\tOAZ3\tODF1\tPACRG\tPAPOLB\tPARP2\tPCSK1N\tPCSK4\tPDHA2\tPEBP1\tPGK2\tPGS1\tPHF7\tPHKG2\tPIAS2\tPOMC\tPRKAR2A\tPRM2\tPSMG1\tRAD17\tRFC4\tRPL39L\tSCG3\tSCG5\tSEPTIN4\tSHE\tSIRT1\tSLC12A2\tSLC2A5\tSNAP91\tSPATA6\tSPMAP2\tSTAM2\tSTRBP\tSYCP1\tTALDO1\tTCP11\tTEKT2\tTKTL1\tTLE4\tTNNI3\tTNP1\tTNP2\tTOPBP1\tTSN\tTSSK2\tTTK\tTUBA3C\tTULP2\tVDAC3\tYBX2\tZC2HC1C\tZC3H14\tZNRF4\tZPBP\nHALLMARK_TGF_BETA_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_TGF_BETA_SIGNALING\tACVR1\tAPC\tARID4B\tBCAR3\tBMP2\tBMPR1A\tBMPR2\tCDH1\tCDK9\tCDKN1C\tCTNNB1\tENG\tFKBP1A\tFNTA\tFURIN\tHDAC1\tHIPK2\tID1\tID2\tID3\tIFNGR2\tJUNB\tKLF10\tLEFTY2\tLTBP2\tMAP3K7\tNCOR2\tNOG\tPMEPA1\tPPM1A\tPPP1CA\tPPP1R15A\tRAB31\tRHOA\tSERPINE1\tSKI\tSKIL\tSLC20A1\tSMAD1\tSMAD3\tSMAD6\tSMAD7\tSMURF1\tSMURF2\tSPTBN1\tTGFB1\tTGFBR1\tTGIF1\tTHBS1\tTJP1\tTRIM33\tUBE2D3\tWWTR1\tXIAP\nHALLMARK_TNFA_SIGNALING_VIA_NFKB\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_TNFA_SIGNALING_VIA_NFKB\tABCA1\tACKR3\tAREG\tATF3\tATP2B1\tB4GALT1\tB4GALT5\tBCL2A1\tBCL3\tBCL6\tBHLHE40\tBIRC2\tBIRC3\tBMP2\tBTG1\tBTG2\tBTG3\tCCL2\tCCL20\tCCL4\tCCL5\tCCN1\tCCND1\tCCNL1\tCCRL2\tCD44\tCD69\tCD80\tCD83\tCDKN1A\tCEBPB\tCEBPD\tCFLAR\tCLCF1\tCSF1\tCSF2\tCXCL1\tCXCL10\tCXCL11\tCXCL2\tCXCL3\tCXCL6\tDENND5A\tDNAJB4\tDRAM1\tDUSP1\tDUSP2\tDUSP4\tDUSP5\tEDN1\tEFNA1\tEGR1\tEGR2\tEGR3\tEHD1\tEIF1\tETS2\tF2RL1\tF3\tFJX1\tFOS\tFOSB\tFOSL1\tFOSL2\tFUT4\tG0S2\tGADD45A\tGADD45B\tGCH1\tGEM\tGFPT2\tGPR183\tHBEGF\tHES1\tICAM1\tICOSLG\tID2\tIER2\tIER3\tIER5\tIFIH1\tIFIT2\tIFNGR2\tIL12B\tIL15RA\tIL18\tIL1A\tIL1B\tIL23A\tIL6\tIL6ST\tIL7R\tINHBA\tIRF1\tIRS2\tJAG1\tJUN\tJUNB\tKDM6B\tKLF10\tKLF2\tKLF4\tKLF6\tKLF9\tKYNU\tLAMB3\tLDLR\tLIF\tLITAF\tMAFF\tMAP2K3\tMAP3K8\tMARCKS\tMCL1\tMSC\tMXD1\tMYC\tNAMPT\tNFAT5\tNFE2L2\tNFIL3\tNFKB1\tNFKB2\tNFKBIA\tNFKBIE\tNINJ1\tNR4A1\tNR4A2\tNR4A3\tOLR1\tPANX1\tPDE4B\tPDLIM5\tPER1\tPFKFB3\tPHLDA1\tPHLDA2\tPLAU\tPLAUR\tPLEK\tPLK2\tPLPP3\tPMEPA1\tPNRC1\tPPP1R15A\tPTGER4\tPTGS2\tPTPRE\tPTX3\tRCAN1\tREL\tRELA\tRELB\tRHOB\tRIGI\tRIPK2\tRNF19B\tSAT1\tSDC4\tSERPINB2\tSERPINB8\tSERPINE1\tSGK1\tSIK1\tSLC16A6\tSLC2A3\tSLC2A6\tSMAD3\tSNN\tSOCS3\tSOD2\tSPHK1\tSPSB1\tSQSTM1\tSTAT5A\tTANK\tTAP1\tTGIF1\tTIPARP\tTLR2\tTNC\tTNF\tTNFAIP2\tTNFAIP3\tTNFAIP6\tTNFAIP8\tTNFRSF9\tTNFSF9\tTNIP1\tTNIP2\tTRAF1\tTRIB1\tTRIP10\tTSC22D1\tTUBB2A\tVEGFA\tYRDC\tZBTB10\tZC3H12A\tZFP36\nHALLMARK_UNFOLDED_PROTEIN_RESPONSE\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_UNFOLDED_PROTEIN_RESPONSE\tALDH18A1\tARFGAP1\tASNS\tATF3\tATF4\tATF6\tATP6V0D1\tBAG3\tBANF1\tCALR\tCCL2\tCEBPB\tCEBPG\tCHAC1\tCKS1B\tCNOT2\tCNOT4\tCNOT6\tCXXC1\tDCP1A\tDCP2\tDCTN1\tDDIT4\tDDX10\tDKC1\tDNAJA4\tDNAJB9\tDNAJC3\tEDC4\tEDEM1\tEEF2\tEIF2AK3\tEIF2S1\tEIF4A1\tEIF4A2\tEIF4A3\tEIF4E\tEIF4EBP1\tEIF4G1\tERN1\tERO1A\tEXOC2\tEXOSC1\tEXOSC10\tEXOSC2\tEXOSC4\tEXOSC5\tEXOSC9\tFKBP14\tFUS\tGEMIN4\tGOSR2\tH2AX\tHERPUD1\tHSP90B1\tHSPA5\tHSPA9\tHYOU1\tIARS1\tIFIT1\tIGFBP1\tIMP3\tKDELR3\tKHSRP\tKIF5B\tLSM1\tLSM4\tMTHFD2\tMTREX\tNABP1\tNFYA\tNFYB\tNHP2\tNOLC1\tNOP14\tNOP56\tNPM1\tPAIP1\tPARN\tPDIA5\tPDIA6\tPOP4\tPREB\tPSAT1\tRPS14\tRRP9\tSDAD1\tSEC11A\tSEC31A\tSERP1\tSHC1\tSKIC3\tSLC1A4\tSLC30A5\tSLC7A5\tSPCS1\tSPCS3\tSRPRA\tSRPRB\tSSR1\tSTC2\tTARS1\tTATDN2\tTSPYL2\tTUBB2A\tVEGFA\tWFS1\tWIPI1\tXBP1\tXPOT\tYIF1A\tYWHAZ\tZBTB17\nHALLMARK_UV_RESPONSE_DN\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_UV_RESPONSE_DN\tABCC1\tACVR2A\tADD3\tADGRL2\tADORA2B\tAGGF1\tAKT3\tAMPH\tANXA2\tANXA4\tAPBB2\tARHGEF9\tATP2B1\tATP2B4\tATP2C1\tATRN\tATRX\tATXN1\tBCKDHB\tBDNF\tBHLHE40\tBMPR1A\tCACNA1A\tCAP2\tCAV1\tCCN1\tCDC42BPA\tCDK13\tCDKN1B\tCDON\tCELF2\tCITED2\tCOL11A1\tCOL1A1\tCOL1A2\tCOL3A1\tCOL5A2\tDAB2\tDBP\tDDAH1\tDLC1\tDLG1\tDMAC2L\tDUSP1\tDYRK1A\tEFEMP1\tERBB2\tF3\tFBLN5\tFHL2\tFYN\tFZD2\tGCNT1\tGJA1\tGRK5\tHAS2\tICA1\tID1\tIGF1R\tIGFBP5\tINPP4B\tINSIG1\tIRS1\tITGB3\tKALRN\tKCNMA1\tKIT\tLAMC1\tLDLR\tLPAR1\tLTBP1\tMAGI2\tMAP1B\tMAP2K5\tMAPK14\tMET\tMGLL\tMGMT\tMIOS\tMMP16\tMRPS31\tMT1E\tMTA1\tMYC\tNEK7\tNFIB\tNFKB1\tNIPBL\tNOTCH2\tNR1D2\tNR3C1\tNRP1\tPDGFRB\tPDLIM5\tPEX14\tPHF3\tPIAS3\tPIK3CD\tPIK3R3\tPLCB4\tPLPP3\tPMP22\tPPARG\tPRDM2\tPRKAR2B\tPRKCA\tPRKCE\tPTEN\tPTGFR\tPTPN21\tPTPRM\tRASA2\tRBPMS\tRGS4\tRND3\tRUNX1\tRXRA\tSCAF8\tSCHIP1\tSCN8A\tSDC2\tSERPINE1\tSFMBT1\tSIPA1L1\tSLC67A1\tSLC7A1\tSMAD3\tSMAD7\tSNAI2\tSPOP\tSRI\tSYNE1\tSYNJ2\tTENT4A\tTFPI\tTGFBR2\tTGFBR3\tTJP1\tTOGARAM1\tVAV2\tVLDLR\tWDR37\tYTHDC1\tZMIZ1\nHALLMARK_UV_RESPONSE_UP\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_UV_RESPONSE_UP\tABCB1\tACAA1\tAGO2\tALAS1\tALDOA\tAMD1\tAP2S1\tAPOM\tAQP3\tARRB2\tASNS\tATF3\tATP6V1C1\tATP6V1F\tBAK1\tBCL2L11\tBID\tBMP2\tBSG\tBTG1\tBTG2\tBTG3\tC4BPB\tCA2\tCASP3\tCCK\tCCND3\tCCNE1\tCDC34\tCDC5L\tCDK2\tCDKN1C\tCDKN2B\tCDO1\tCEBPG\tCHKA\tCHRNA5\tCLCN2\tCLTB\tCNP\tCOL2A1\tCREG1\tCTSV\tCXCL2\tCYB5B\tCYB5R1\tCYP1A1\tDDX21\tDGAT1\tDLG4\tDNAJA1\tDNAJB1\tE2F5\tEIF2S3\tEIF5\tENO2\tEPCAM\tEPHX1\tFEN1\tFGF18\tFKBP4\tFMO1\tFOS\tFOSB\tFURIN\tGAL\tGCH1\tGGH\tGLS\tGPX3\tGRINA\tGRPEL1\tH2AX\tHLA-F\tHMOX1\tHNRNPU\tHSPA13\tHSPA2\tHTR7\tHYAL2\tICAM1\tIGFBP2\tIL6\tIL6ST\tIRF1\tJUNB\tKCNH2\tKLHDC3\tLHX2\tLYN\tMAOA\tMAPK8IP2\tMARK2\tMGAT1\tMMP14\tMRPL23\tMSX1\tNAT1\tNFKBIA\tNKX2-5\tNPTX2\tNPTXR\tNR4A1\tNTRK3\tNUP58\tNXF1\tOLFM1\tONECUT1\tPARP2\tPDAP1\tPDLIM3\tPLCL1\tPOLE3\tPOLG2\tPOLR2H\tPPAT\tPPIF\tPPP1R2\tPPT1\tPRKACA\tPRKCD\tPRPF3\tPSMC3\tPTPRD\tRAB27A\tRASGRP1\tRET\tRFC4\tRHOB\tRPN1\tRRAD\tRXRB\tSELENOW\tSHOX2\tSIGMAR1\tSLC25A4\tSLC6A12\tSLC6A8\tSOD2\tSPR\tSQSTM1\tSTARD3\tSTIP1\tSTK25\tSULT1A1\tTACR3\tTAP1\tTARS1\tTCHH\tTFRC\tTGFBRAP1\tTMBIM6\tTST\tTUBA4A\tTYRO3\tUROD\tWIZ\tYKT6\nHALLMARK_WNT_BETA_CATENIN_SIGNALING\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_WNT_BETA_CATENIN_SIGNALING\tADAM17\tAXIN1\tAXIN2\tCCND2\tCSNK1E\tCTNNB1\tCUL1\tDKK1\tDKK4\tDLL1\tDVL2\tFRAT1\tFZD1\tFZD8\tGNAI1\tHDAC11\tHDAC2\tHDAC5\tHEY1\tHEY2\tJAG1\tJAG2\tKAT2A\tLEF1\tMAML1\tMYC\tNCOR2\tNCSTN\tNKD1\tNOTCH1\tNOTCH4\tNUMB\tPPARD\tPSEN2\tPTCH1\tRBPJ\tSKP2\tTCF7\tTP53\tWNT1\tWNT5B\tWNT6\nHALLMARK_XENOBIOTIC_METABOLISM\thttps://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_XENOBIOTIC_METABOLISM\tABCC2\tABCC3\tABCD2\tABHD6\tACO2\tACOX1\tACOX2\tACOX3\tACP1\tACP2\tACSM1\tADH1C\tADH5\tADH7\tAHCY\tAKR1C2\tAKR1C3\tALAS1\tALDH2\tALDH3A1\tALDH9A1\tANGPTL3\tAOX1\tAP4B1\tAPOE\tAQP9\tARG1\tARG2\tARPP19\tASL\tATOH8\tATP2A2\tBCAR1\tBCAT1\tBLVRB\tBPHL\tCA2\tCASP6\tCAT\tCBR1\tCCL25\tCD36\tCDA\tCDO1\tCES1\tCFB\tCNDP2\tCOMT\tCROT\tCRP\tCSAD\tCYB5A\tCYFIP2\tCYP17A1\tCYP1A1\tCYP1A2\tCYP26A1\tCYP27A1\tCYP2C18\tCYP2E1\tCYP2J2\tCYP2S1\tCYP4F2\tDCXR\tDDAH2\tDDC\tDDT\tDHPS\tDHRS1\tDHRS7\tECH1\tELOVL5\tENPEP\tENTPD5\tEPHA2\tEPHX1\tESR1\tETFDH\tETS2\tF10\tF11\tFABP1\tFAH\tFAS\tFBLN1\tFBP1\tFETUB\tFMO1\tFMO3\tG6PC1\tGABARAPL1\tGAD1\tGART\tGCH1\tGCKR\tGCLC\tGCNT2\tGNMT\tGSR\tGSS\tGSTA3\tGSTM4\tGSTO1\tGSTT2\tHACL1\tHES6\tHGFAC\tHMOX1\tHNF4A\tHPRT1\tHRG\tHSD11B1\tHSD17B2\tID2\tIDH1\tIGF1\tIGFBP1\tIGFBP4\tIL1R1\tIRF8\tITIH1\tITIH4\tJUP\tKARS1\tKYNU\tLCAT\tLEAP2\tLONP1\tLPIN2\tMAN1A1\tMAOA\tMARCHF6\tMBL2\tMCCC2\tMPP2\tMT2A\tMTHFD1\tNDRG2\tNFS1\tNINJ1\tNMT1\tNPC1\tNQO1\tPAPSS2\tPC\tPDK4\tPDLIM5\tPEMT\tPGD\tPGRMC1\tPINK1\tPLG\tPMM1\tPOR\tPPARD\tPROS1\tPSMB10\tPTGDS\tPTGES\tPTGES3\tPTGR1\tPTS\tPYCR1\tRAP1GAP\tRBP4\tREG1A\tRETSAT\tSAR1B\tSERPINA6\tSERPINE1\tSERTAD1\tSHMT2\tSLC12A4\tSLC1A5\tSLC22A1\tSLC35B1\tSLC35D1\tSLC46A3\tSLC6A12\tSLC6A6\tSMOX\tSPINT2\tSSR3\tTAT\tTDO2\tTGFB2\tTKFC\tTMBIM6\tTMEM176B\tTMEM97\tTNFRSF1A\tTPST1\tTTPA\tTYR\tUGDH\tUPB1\tUPP1\tVNN1\tVTN\tXDH\n'

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, shutil, time, traceback, uuid, zipfile
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive')
INPUT = BASE/'SpaGAT_revision/R3_5_7_gene_signed_20260928_090940_UTC'
RECEIVERS = ['tumor_1', 'tumor_2']
MIN_PROGRAM_GENES = 5
BLOCK = 2048
GMT_URL = 'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2026.1.Hs/h.all.v2026.1.Hs.symbols.gmt'
GMT_SHA = 'eecaf6dad908334ae885406ec72bdc0646d8917588ed7c219fac92fc5363f596'


def save_json(p, data):
    Path(p).write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str))


def sha(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda: f.read(4*1024*1024), b''):
            h.update(b)
    return h.hexdigest()


def stage(src, cache, manifest):
    src = Path(src)
    if not src.is_file():
        raise FileNotFoundError(f'缺少已保存数组：{src}；不会重新推理。')
    dest = cache/src.name
    h = hashlib.sha256()
    with src.open('rb') as a, dest.open('xb') as b:
        for chunk in iter(lambda: a.read(4*1024*1024), b''):
            b.write(chunk)
            h.update(chunk)
    manifest.append(dict(path=str(src), bytes=src.stat().st_size, sha256=h.hexdigest()))
    return np.load(dest, mmap_mode='r', allow_pickle=False)


def read_sets(text, genes):
    sets = {}
    for line in text.splitlines():
        row = line.split('\t')
        if len(row) < 3 or row[0] in sets:
            raise ValueError('Invalid or duplicate GMT row')
        sets[row[0]] = dict(url=row[1], genes=sorted(set(row[2:])))
    universe = set(g for s in sets.values() for g in s['genes'])
    # Only exact symbols or unambiguous hyphen-to-dot column-name transformations.
    # No fuzzy matching or inferred aliases.
    aliases = {}
    for symbol in universe:
        aliases.setdefault(symbol.replace('-', '.'), set()).add(symbol)
    mapping = []
    for gene in genes:
        candidates = aliases.get(gene, set())
        mapped = gene if gene in universe else next(iter(candidates)) if len(candidates) == 1 else None
        how = 'exact' if gene in universe else 'unique_hyphen_to_dot' if mapped else 'not_in_library_or_ambiguous'
        mapping.append(dict(panel_gene=gene, library_symbol=mapped, mapping=how))
    used = [x['library_symbol'] for x in mapping if x['library_symbol']]
    if len(used) != len(set(used)):
        raise ValueError('Multiple panel columns map to the same biological symbol')
    rows, members = [], {}
    for name, item in sorted(sets.items()):
        inds = [i for i, m in enumerate(mapping) if m['library_symbol'] in item['genes']]
        members[name] = inds
        rows.append(dict(program=name, full_set_size=len(item['genes']), panel_genes=len(inds),
                         panel_coverage=len(inds)/len(item['genes']), gene_set_url=item['url'],
                         panel_gene_list=';'.join(genes[i] for i in inds),
                         library_gene_list=';'.join(mapping[i]['library_symbol'] for i in inds)))
    return pd.DataFrame(mapping), pd.DataFrame(rows), members


def moments(pred, obs, indices):
    n, g = len(indices), pred.shape[1]
    if n < 2:
        raise ValueError('Need at least two cells')
    sx = np.zeros(g); sy = np.zeros(g); err = np.zeros(g); zero = np.zeros(g)
    neg = np.zeros(g, dtype=np.int64)
    for start in range(0, n, BLOCK):
        ids = indices[start:start+BLOCK]
        p, y = np.asarray(pred[ids], float), np.asarray(obs[ids], float)
        if not np.isfinite(p).all() or not np.isfinite(y).all():
            raise ValueError('Nonfinite predictions or targets')
        sx += p.sum(0); sy += y.sum(0); err += ((p-y)**2).sum(0); zero += (y*y).sum(0)
        neg += (y < 0).sum(0)
    mp, my = sx/n, sy/n
    vp = np.zeros(g); vy = np.zeros(g); cov = np.zeros(g)
    for start in range(0, n, BLOCK):
        ids = indices[start:start+BLOCK]
        p, y = np.asarray(pred[ids], float)-mp, np.asarray(obs[ids], float)-my
        vp += (p*p).sum(0); vy += (y*y).sum(0); cov += (p*y).sum(0)
    den = np.sqrt(vp*vy)
    pcc = np.divide(cov, den, out=np.full(g, np.nan), where=den>0)
    return dict(n=n, pred_mean=mp, obs_mean=my, pred_var=vp/n, obs_var=vy/n,
                MSE=err/n, MSE0=zero/n, PCC=pcc, eps_PCC=cov/(den+1e-8),
                negative_fraction=neg/n)


def weights_for_programs(m, members):
    names = sorted(members)
    raw = np.zeros((len(m['PCC']), len(names)))
    z = np.zeros_like(raw)
    sd = np.sqrt(m['obs_var'])
    valid_sd = np.isfinite(sd) & (sd > 0)
    details = []
    for k, name in enumerate(names):
        ids = np.asarray(members[name], dtype=int)
        usable = ids[valid_sd[ids]]
        raw_ok, z_ok = len(ids) >= MIN_PROGRAM_GENES, len(usable) >= MIN_PROGRAM_GENES
        if raw_ok:
            raw[ids, k] = 1/len(ids)
        if z_ok:
            z[usable, k] = 1/(len(usable)*sd[usable])
        details.append(dict(program=name, raw_evaluable=raw_ok, standardized_evaluable=z_ok,
                            standardized_genes=len(usable), excluded_zero_variance=len(ids)-len(usable),
                            passes_10_gene_sensitivity=len(usable)>=10))
    return names, raw, z, pd.DataFrame(details)


def project(arr, indices, weights):
    result = np.empty((len(indices), weights.shape[1]), dtype=np.float64)
    for start in range(0, len(indices), BLOCK):
        result[start:start+BLOCK] = np.asarray(arr[indices[start:start+BLOCK]], float) @ weights
    return result


def correlations(p, y):
    a, b = p-p.mean(), y-y.mean()
    den = np.sqrt(np.dot(a, a)*np.dot(b, b))
    return float(np.dot(a, b)/den) if den > 0 else np.nan


def score_metrics(p, y):
    mse = float(np.mean((p-y)**2)); mse0 = float(np.mean(y*y))
    # Inputs share observed-derived centering/scaling; predictions are not restandardized.
    nz = np.abs(y)>1e-12
    return dict(PCC=correlations(p,y),
                Spearman=correlations(pd.Series(p).rank().to_numpy(),pd.Series(y).rank().to_numpy()),
                MSE=mse, MSE0=mse0, EV_percent=100*(1-mse/mse0) if mse0>0 else np.nan,
                observed_mean=float(y.mean()), predicted_mean=float(p.mean()),
                observed_variance=float(y.var()), predicted_variance=float(p.var()),
                direction_agreement=float(np.mean(np.sign(p[nz])==np.sign(y[nz]))) if nz.any() else np.nan,
                direction_n=int(nz.sum()))


def program_recovery(pred, obs, meta, genes, members, coverage, out):
    metrics, audits, coverage_rows, scalings, receiver_models = [], [], [], [], {}
    for receiver in RECEIVERS:
        idx = np.flatnonzero(meta.cell_type.to_numpy()==receiver)
        m = moments(pred, obs, idx)
        old = pd.read_csv(INPUT/f'{receiver}_gene_recovery.csv').set_index('gene').loc[genes]
        for col, values in [('PCC',m['PCC']),('MSE',m['MSE']),('observed_mean',m['obs_mean'])]:
            if not np.allclose(old[col], values, rtol=1e-5, atol=1e-7, equal_nan=True):
                raise ValueError(f'{receiver} 数组与先前基因表不一致：{col}')
        pd.DataFrame(dict(gene=genes, observed_mean=m['obs_mean'], observed_sd=np.sqrt(m['obs_var']),
                          observed_negative_fraction=m['negative_fraction'])).to_csv(out/f'{receiver}_target_diagnostics.csv',index=False)
        names, raw, z, detail = weights_for_programs(m, members)
        info = coverage.merge(detail,on='program'); info.insert(0,'receiver',receiver); coverage_rows.append(info)
        scalings.append(pd.DataFrame(dict(receiver=receiver,gene=genes,observed_mean=m['obs_mean'],observed_sd=np.sqrt(m['obs_var']))))
        for label, W, center, valid_col in [('observed_z_mean',z,m['obs_mean']@z,'standardized_evaluable'),
                                          ('raw_mean',raw,np.zeros(len(names)),'raw_evaluable')]:
            yp = project(pred,idx,W)-center
            yt = project(obs,idx,W)-center
            for k,name in enumerate(names):
                valid = bool(detail.loc[k,valid_col])
                row = dict(receiver=receiver, program=name, score=label, n_cells=len(idx), evaluable=valid,
                           direction_reference='observed_population_mean' if label=='observed_z_mean' else 'original_target_zero')
                if valid:
                    row.update(score_metrics(yp[:,k],yt[:,k]))
                metrics.append(row)
            # Save modest score arrays in the new Drive folder, not the downloadable ZIP.
            np.savez_compressed(out/'score_arrays'/f'{receiver}_{label}.npz',
                                prediction=yp.astype('float32'),observed=yt.astype('float32'),
                                dataset_indices=meta.dataset_index.to_numpy()[idx],programs=np.asarray(names))
        receiver_models[receiver]=(names,z,detail)
        manuscript={'tumor_1':.267,'tumor_2':.277}[receiver]
        for subset, inds in [('all_validation',idx),('both_senders_present',idx[meta.masking_eligible.to_numpy()[idx]])]:
            mm=m if subset=='all_validation' else moments(pred,obs,inds)
            for rule,vals in [('Pearson_float64',mm['PCC']),('denominator_plus_1e-8',mm['eps_PCC'])]:
                vals=vals[np.isfinite(vals)]
                for median_rule,v in [('numpy_median',np.median(vals)),('lower_middle_like_torch',np.sort(vals)[(len(vals)-1)//2])]:
                    audits.append(dict(receiver=receiver,subset=subset,n_cells=len(inds),PCC_rule=rule,
                                       median_rule=median_rule,median_PCC=v,manuscript_Figure4=manuscript,
                                       difference_from_manuscript=v-manuscript))
        print(receiver, 'program recovery 完成；median gene PCC =', round(float(np.nanmedian(m['PCC'])),6),flush=True)
    result=pd.DataFrame(metrics)
    result.to_csv(out/'program_recovery.csv',index=False)
    pd.concat(coverage_rows).to_csv(out/'program_coverage.csv',index=False)
    pd.concat(scalings).to_csv(out/'program_score_scaling.csv',index=False)
    pd.DataFrame(audits).to_csv(out/'Figure4_metric_audit.csv',index=False)
    return receiver_models, result


def signed_programs(meta, genes, models, out, cache, manifest):
    repeats=pd.read_csv(INPUT/'signed_masking_gene_repeats.csv')
    if repeats.duplicated(['receiver','condition','repeat','gene']).any():
        raise ValueError('Duplicate signed masking records')
    program_repeats=[]; cellwise=[]
    for receiver in RECEIVERS:
        names,W,detail=models[receiver]
        wanted=meta[(meta.cell_type==receiver)&meta.masking_eligible].dataset_index.to_numpy()
        for condition in ['tumor_1','tumor_2','random']:
            prefix=f'{receiver}__{condition}'
            ids=np.load(INPUT/'arrays'/f'{prefix}__dataset_indices.npy',allow_pickle=False)
            if not np.array_equal(ids,wanted):
                raise ValueError('Signed array receiver order mismatch')
            arr=stage(INPUT/'arrays'/f'{prefix}__mean_delta.npy',cache,manifest)
            if arr.shape!=(len(wanted),len(genes)):
                raise ValueError('Signed array shape mismatch')
            scores=project(arr,np.arange(len(arr)),W)
            rt=repeats[(repeats.receiver==receiver)&(repeats.condition==condition)]
            table=rt.pivot(index='repeat',columns='gene',values='mean_signed_delta').reindex(columns=genes)
            if table.shape!=(10,len(genes)) or not np.isfinite(table.to_numpy()).all():
                raise ValueError('Missing repeat/gene combinations')
            projected=table.to_numpy()@W
            if not np.allclose(scores.mean(0),projected.mean(0),rtol=2e-5,atol=2e-6):
                raise ValueError('Program signed mean does not reconcile with gene-level repeat means')
            for k,name in enumerate(names):
                if not detail.loc[k,'standardized_evaluable']:
                    continue
                for rep,val in zip(table.index,projected[:,k]):
                    program_repeats.append(dict(receiver=receiver,condition=condition,repeat=int(rep),program=name,
                                                n_receivers=len(ids),mean_signed_score_delta=float(val)))
                cellwise.append(dict(receiver=receiver,condition=condition,program=name,n_receivers=len(ids),
                    mean_absolute_of_cellwise_repeat_mean_delta=float(np.abs(scores[:,k]).mean()),
                    fraction_cells_positive_repeat_mean_delta=float((scores[:,k]>0).mean()),
                    fraction_cells_negative_repeat_mean_delta=float((scores[:,k]<0).mean())))
            del arr,scores
        print(receiver,'signed program summary 完成',flush=True)
    rt=pd.DataFrame(program_repeats)
    rt.to_csv(out/'signed_program_repeats.csv',index=False)
    s=rt.groupby(['receiver','condition','program']).agg(
        n_receivers=('n_receivers','first'),mean_signed_score_delta=('mean_signed_score_delta','mean'),
        SD_across_masking_repeats=('mean_signed_score_delta','std'),
        min_repeat_mean=('mean_signed_score_delta','min'),max_repeat_mean=('mean_signed_score_delta','max')).reset_index()
    s=s.merge(pd.DataFrame(cellwise),on=['receiver','condition','program','n_receivers'])
    s['same_sign_in_all_repeat_means']=(s.min_repeat_mean>0)|(s.max_repeat_mean<0)
    s.to_csv(out/'signed_program_summary.csv',index=False)
    random=rt[rt.condition=='random'][['receiver','repeat','program','mean_signed_score_delta']].rename(columns={'mean_signed_score_delta':'random_delta'})
    control=rt[rt.condition!='random'].merge(random,on=['receiver','repeat','program'],validate='many_to_one')
    control['signed_delta_minus_random']=control.mean_signed_score_delta-control.random_delta
    control.to_csv(out/'signed_program_minus_random_repeats.csv',index=False)
    control.groupby(['receiver','condition','program']).signed_delta_minus_random.agg(['mean','std']).reset_index().to_csv(out/'signed_program_minus_random_summary.csv',index=False)
    # Identical-count random control is descriptive. Differences are not isolated causal effects.
    for receiver in RECEIVERS:
        rec=pd.read_csv(INPUT/f'{receiver}_gene_recovery.csv')
        a=pd.read_csv(INPUT/'signed_masking_gene_summary.csv')
        a=a[a.receiver==receiver].merge(rec[['gene','PCC','EV_percent']],on='gene',validate='many_to_one')
        rand=a[a.condition=='random'][['gene','mean_signed_delta','mean_absolute_delta']].rename(columns={
            'mean_signed_delta':'random_mean_signed_delta','mean_absolute_delta':'random_mean_absolute_delta'})
        a=a.merge(rand,on='gene',validate='many_to_one')
        a['signed_delta_minus_random']=a.mean_signed_delta-a.random_mean_signed_delta
        a['absolute_delta_minus_random']=a.mean_absolute_delta-a.random_mean_absolute_delta
        a['rank_mean_absolute_delta']=a.groupby('condition').mean_absolute_delta.rank(ascending=False,method='min').astype(int)
        a.sort_values(['condition','rank_mean_absolute_delta','gene']).to_csv(out/f'{receiver}_signed_gene_ranked.csv',index=False)
    return s


def evidence_audit(provenance, genes, out):
    """Read small project records/source only; do not execute discovered source."""
    start=time.monotonic(); visited=0; snippets=[]; records=[]; errors=[]; baselines=[]
    limits=dict(max_files=2500,max_seconds=90,max_source_bytes=1500000,max_csv_bytes=3000000,
                max_snippets=120,max_source_copies=15)
    source_rx=re.compile(r'typeexp|residual|groupby|subtract|log1p|log2|normalize_total|tumor_1|0\.267|0\.277',re.I)
    csv_rx=re.compile(r'liver|tumor|cell.?type|receiver|pcc|metric|figure.?4',re.I)
    source_dir=out/'source_evidence';source_dir.mkdir()
    candidates=[]
    for root in [BASE/'spagatv2',BASE/'SpaGAT_submission',BASE/'GITIII-main/liver_workdir']:
        if not root.exists():
            errors.append(dict(path=str(root),error='missing search root'));continue
        for d,dirs,files in os.walk(root):
            dirs[:]=sorted(x for x in dirs if x not in {'.git','__pycache__','.ipynb_checkpoints','node_modules','venv','.venv'})
            for name in sorted(files):
                visited+=1
                if visited>limits['max_files'] or time.monotonic()-start>limits['max_seconds']:
                    break
                p=Path(d)/name
                try:
                    size=p.stat().st_size
                    if p.suffix in {'.py','.ipynb'} and size<limits['max_source_bytes']:
                        text=p.read_text(errors='replace')
                        if p.suffix=='.ipynb':
                            nb=json.loads(text)
                            text='\n'.join(''.join(c.get('source',[])) for c in nb.get('cells',[]) if c.get('cell_type')=='code')
                        if not source_rx.search(text):continue
                        lines=text.splitlines()
                        hits=[i for i,l in enumerate(lines) if source_rx.search(l)]
                        strong=('TypeExp' in text and any(x in text for x in ['groupby','subtract',' - ','-=']))
                        candidates.append((strong,p,text))
                        for i in hits[:6]:
                            if len(snippets)<limits['max_snippets']:
                                snippets.append(dict(path=str(p),line=i+1,line_kind='flattened_code' if p.suffix=='.ipynb' else 'source',
                                    excerpt='\n'.join(lines[max(0,i-3):min(len(lines),i+4)])))
                    elif p.suffix=='.csv' and size<limits['max_csv_bytes'] and csv_rx.search(str(p.relative_to(root))):
                        df=pd.read_csv(p,nrows=30000)
                        cols=[c for c in df if re.search('pcc|correl',str(c),re.I)]
                        labelcols=[c for c in df if re.search('cell.?type|receiver|population|subclass',str(c),re.I)]
                        if not cols or not labelcols:continue
                        selected=np.zeros(len(df),dtype=bool)
                        for c in labelcols:selected|=df[c].astype(str).str.contains('tumor',case=False,regex=False).to_numpy()
                        for _,row in df[selected].head(100).iterrows():
                            records.append(dict(path=str(p),sha256=sha(p),values=row.to_dict()))
                except Exception as e:errors.append(dict(path=str(p),error=str(e)))
            if visited>limits['max_files'] or time.monotonic()-start>limits['max_seconds']:break
        if visited>limits['max_files'] or time.monotonic()-start>limits['max_seconds']:break
    copies=[]
    for i,(_,p,text) in enumerate(sorted(candidates,key=lambda x:(not x[0],str(x[1])))[:limits['max_source_copies']]):
        dest=source_dir/f'{i:02d}_{p.stem}.txt';dest.write_text(text)
        copies.append(dict(path=str(p),sha256=sha(p),copied_as=str(dest.relative_to(out))))
    baseline_path=Path(provenance['data_dir'])/'CancerousLiver_TypeExp.npz'
    if baseline_path.exists():
        try:
            with np.load(baseline_path,allow_pickle=False) as z:
                vals={t:np.asarray(z[t]) for t in RECEIVERS}
                if any(v.shape!=(len(genes),) or not np.isfinite(v).all() for v in vals.values()):
                    raise ValueError('Baseline shape or values invalid')
                frame=pd.DataFrame(dict(gene=genes,tumor_1_supplied_baseline=vals['tumor_1'],
                    tumor_2_supplied_baseline=vals['tumor_2'],baseline_tumor2_minus_tumor1=vals['tumor_2']-vals['tumor_1']))
                frame.to_csv(out/'supplied_tumor_baseline_profiles.csv',index=False)
            baselines.append(dict(path=str(baseline_path),sha256=sha(baseline_path),
                interpretation='Supplied baseline vectors; transformation and estimation population require source verification. Not a validated tumor subtype assignment.'))
        except Exception as e:errors.append(dict(path=str(baseline_path),error=str(e)))
    save_json(out/'source_snippets.json',snippets)
    save_json(out/'historical_tumor_metric_records.json',records)
    save_json(out/'preprocessing_and_history_audit.json',dict(limits=limits,visited_files=visited,
        incomplete=visited>limits['max_files'] or time.monotonic()-start>limits['max_seconds'],
        source_copies=copies,baseline_evidence=baselines,errors=errors,
        target_semantics='PENDING_MANUAL_SOURCE_VERIFICATION',
        Figure4_origin='PENDING_RECONCILIATION; numerical agreement alone does not prove identical checkpoint or evaluated cells'))
    print('源码和历史结果线索已收集；不会自动把线索标为预处理来源已验证。',flush=True)


def scalar_comparison(out):
    old=BASE/'spagatv2/analysis/liver_sender_origin_random_control/matched_sender_random_control_repeats.csv'
    if not old.exists():return
    keys=['receiver','condition']
    cols=['full_MSE','masked_MSE','mean_abs_prediction_change','relative_MSE_increase_percent']
    a=pd.read_csv(old).groupby(keys)[cols].mean()
    b=pd.read_csv(INPUT/'masking_scalar_repeats.csv').groupby(keys)[cols].mean()
    merged=a.join(b,lsuffix='_historical',rsuffix='_new',how='outer')
    for c in cols:merged[c+'_difference']=merged[c+'_new']-merged[c+'_historical']
    merged.reset_index().to_csv(out/'historical_vs_new_masking.csv',index=False)


METHODS = '''Analysis scope: existing validation predictions and fixed-model matched-count masking.
No model is loaded, trained, or run. No independent liver test or biological replication is added.
Gene sets: all 50 human MSigDB Hallmark sets, v2026.1.Hs, downloaded from the cited Broad Institute release.
Use exact symbol matching and only unambiguous hyphen-to-dot column-name mapping. Export all mappings and coverage.
Each program is evaluated if at least 5 panel genes are available; the standardized score also needs at least 5 positive-variance genes. Report all 50 sets, including non-evaluable sets. The >=10-gene indicator is a prespecified sensitivity subset, not a significance threshold. Sparse coverage does not validate a whole pathway.
Primary score within each annotated tumor population: mean of (gene value - observed validation mean)/observed validation SD over available program genes. Exactly the same observed means and SDs are applied to predictions. No prediction-specific rescaling. These descriptive validation-derived calibration constants are exported; this is not an independent-test estimate.
Sensitivity score: unscaled mean of measured target or predicted values for the same gene set. This secondary score includes constant genes, if any. No second baseline subtraction is performed.
Report PCC, Spearman, MSE, MSE0 and 100*(1-MSE/MSE0). For the standardized score, zero is the observed-mean program baseline. For the raw score, zero is the original target-zero baseline; its residual interpretation remains conditional on preprocessing provenance.
Direction agreement compares the signs of standardized predicted and observed scores, excluding only |observed score|<=1e-12. For the standardized score this is direction around the observed population mean; for the raw score the reference is original target zero, as explicitly labeled. Neither is differential expression between tumor populations or tumor versus normal.
Signed delta = masked prediction - full prediction. Program signed means for each masking repeat are exact linear combinations of existing gene-level repeat means. Repeat SD describes neighbor selection under a fixed model, not training variability or biological replication.
Cell-level program deltas are projected from arrays already averaged across 10 repeats. Mean absolute of these cell-level averages is |mean over repeats(delta)| averaged over cells. It is NOT mean over repeats and cells(|delta|). This distinction is reflected in column names.
Random-control subtraction is descriptive under equal removed counts; it does not remove all spatial, abundance, or graph-perturbation confounding.
The source audit gathers candidate preprocessing code and historical metric records only. Near-zero target means, numerical agreement, or a code keyword does not by itself verify historical provenance. Residual target semantics, old Figure 4 identity and tumor molecular subtype identity remain subject to manual evidence review.
Supplied tumor baseline vectors are exported for annotation follow-up, without inferring subtype identity or independent disease validation.
No gene-set significance tests, cell-bootstrap confidence intervals, clinical subtype assignments, or causal claims are generated.
'''


def run_followup():
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')+'_'+uuid.uuid4().hex[:6]
    out=BASE/'SpaGAT_revision'/('R3_5_7_program_followup_'+stamp)
    out.mkdir(parents=True,exist_ok=False)
    cache=Path('/content')/('R3_program_cache_'+stamp);cache.mkdir(exist_ok=False)
    manifest=[]; status=dict(status='running',new_training=False,new_inference=False,input=str(INPUT),
                            gene_set_source=GMT_URL,gene_set_sha256=GMT_SHA,min_program_genes=MIN_PROGRAM_GENES)
    try:
        if not INPUT.is_dir():raise FileNotFoundError(f'找不到原输出目录：{INPUT}。请检查 Drive 挂载和路径。')
        prov=json.loads((INPUT/'provenance.json').read_text())
        if prov['status']!='complete_gene_and_matched_count_masking':raise ValueError('Prior export did not complete')
        genes=json.loads((INPUT/'genes.json').read_text())
        meta=pd.read_csv(INPUT/'receiver_metadata.csv')
        if len(genes)!=1000 or len(set(genes))!=1000:raise ValueError('Unexpected gene panel')
        if meta.dataset_index.duplicated().any() or set(meta.cell_type)!=set(RECEIVERS):raise ValueError('Invalid receiver metadata')
        if not meta.masking_eligible.isin([True,False]).all():raise ValueError('Nonboolean masking eligibility')
        idx=np.load(INPUT/'arrays/dataset_indices.npy',allow_pickle=False)
        if not np.array_equal(idx,meta.dataset_index.to_numpy()):raise ValueError('Prediction row IDs do not match metadata')
        counts=meta.groupby('cell_type').agg(n=('dataset_index','size'),eligible=('masking_eligible','sum')).to_dict('index')
        if counts!={'tumor_1':{'n':69784,'eligible':19659},'tumor_2':{'n':6969,'eligible':6892}}:
            raise ValueError(f'Unexpected receiver counts: {counts}')
        status.update(receiver_counts=counts,selected_checkpoint=prov['selected_checkpoint'],prior_provenance=prov,
            input_metadata_hashes={name:sha(INPUT/name) for name in ['provenance.json','genes.json','receiver_metadata.csv',
                'tumor_1_gene_recovery.csv','tumor_2_gene_recovery.csv','signed_masking_gene_repeats.csv','signed_masking_gene_summary.csv']})
        if hashlib.sha256(GMT_TEXT.encode()).hexdigest()!=GMT_SHA:raise ValueError('Embedded gene-set checksum mismatch')
        (out/'h.all.v2026.1.Hs.symbols.gmt').write_text(GMT_TEXT)
        mapping,coverage,members=read_sets(GMT_TEXT,genes)
        if len(members)!=50:raise ValueError('Expected all 50 Hallmark gene sets')
        mapping.to_csv(out/'gene_symbol_mapping.csv',index=False)
        coverage.to_csv(out/'panel_program_coverage.csv',index=False)
        (out/'METHODS_AND_LIMITS.txt').write_text(METHODS+'\nGene-set source: '+GMT_URL+'\n')
        (out/'score_arrays').mkdir()
        sizes=sum(p.stat().st_size for p in (INPUT/'arrays').glob('*.npy'))
        if shutil.disk_usage(cache).free<sizes+512*1024**2:raise RuntimeError('Colab local disk has insufficient space')
        print('读取已保存的 prediction/target；不加载模型。',flush=True)
        pred=stage(INPUT/'arrays/prediction.npy',cache,manifest)
        obs=stage(INPUT/'arrays/target.npy',cache,manifest)
        if pred.shape!=obs.shape or pred.shape!=(len(meta),len(genes)):raise ValueError('Prediction/target shape mismatch')
        models,result=program_recovery(pred,obs,meta,genes,members,coverage,out)
        signed=signed_programs(meta,genes,models,out,cache,manifest)
        scalar_comparison(out)
        evidence_audit(prov,genes,out)
        status.update(status='complete_postprocessing_pending_provenance_review',
            target_semantics='PENDING_MANUAL_SOURCE_VERIFICATION',Figure4_origin='PENDING_RECONCILIATION',
            tumor_subtype_assignment='NOT_PERFORMED',independent_validation='NOT_PERFORMED')
    except Exception as e:
        status.update(status='stopped',error=str(e))
        (out/'error_trace.txt').write_text(traceback.format_exc())
        print('停止：',e,flush=True)
    finally:
        status['input_arrays']=manifest
        save_json(out/'provenance.json',status)
        archive=out.with_suffix('.zip')
        with zipfile.ZipFile(archive,'x',zipfile.ZIP_DEFLATED) as z:
            for p in sorted(out.rglob('*')):
                if p.is_file() and 'score_arrays' not in p.relative_to(out).parts:
                    z.write(p,out.name+'/'+str(p.relative_to(out)))
        print('运行状态：',status['status'])
        print('请下载并上传这个 ZIP：',archive,flush=True)
    return str(archive)


## 3. 执行并下载 ZIP

结果写入新的 Drive 目录 `SpaGAT_revision/R3_5_7_program_followup_时间戳_随机编号/`。
程序会在每个 tumor 群体完成后打印进度。下载 ZIP 后上传回对话即可，无需上传大型数组。

如果数组缺失、基因顺序或细胞索引不一致，程序会停止并输出诊断 ZIP；不要删除检查强行继续。源代码搜索只是收集证据，不保证自动找齐历史来源。


In [ ]:
archive_path = run_followup()
from google.colab import files
files.download(archive_path)


## 输出文件

- `program_recovery.csv`：两个群体、全部程序、两种 score 的恢复结果，包含不满足覆盖要求的程序。
- `program_coverage.csv` / `gene_symbol_mapping.csv`：覆盖度、实际使用基因、符号映射及恒定基因处理。
- `program_score_scaling.csv`：observed-derived 标准化常数，便于复核。
- `signed_program_summary.csv` / `signed_program_repeats.csv`：程序层面的有符号 masking 结果和重复变异。
- `signed_program_minus_random_summary.csv`：程序有符号变化与同数量随机屏蔽的描述性对比。
- `tumor_*_signed_gene_ranked.csv`：所有基因按绝对变化排序，同时保留符号、预测 PCC 和随机对照。
- `Figure4_metric_audit.csv`：不同评估细胞子集和数值处理的结果与正文数值对照。
- `historical_vs_new_masking.csv`：历史与本次 scalar masking 指标对照，差异原因不自动归因于随机性。
- `historical_tumor_metric_records.json` / `source_evidence/`：旧指标和预处理来源线索。
- `supplied_tumor_baseline_profiles.csv`：所提供的 tumor baseline profiles，需结合预处理/注释来源解释。
- `provenance.json` / `METHODS_AND_LIMITS.txt`：输入校验、基因集版本、方法和仍未完成的核对事项。

`score_arrays/` 留在 Drive 新输出目录，不放入下载 ZIP。此次后处理没有补充独立患者、病理学验证、annotation sensitivity retraining 或 distance-matched controls。
